# FLIP — Kaggle Notebook

Official implementation of [FLIP](https://arxiv.org/abs/2310.18933) (NeurIPS 2023) bundled into a single Kaggle-runnable notebook.

This notebook reproduces the original project layout under the Kaggle working
directory (`/kaggle/working`) by writing each project file via `%%writefile` cells
and then invoking the original entry point `run_experiment.py` exactly as the
original project does locally.

No project logic, hyperparameters, or execution order has been changed.


## 1. Install dependencies


In [ ]:
!pip install -q 'toml>=0.10.0,<0.11.0' 'tqdm>=4.60.0' 'numpy' 'scipy' 'pandas' 'pillow'


## 2. Prepare project working directory on Kaggle


In [ ]:
import os

# Kaggle's writable working directory.
PROJECT_DIR = '/kaggle/working/FLIP'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())


In [ ]:
import os

# Create all necessary subdirectories matching the original project layout.
for d in [
    'data',
    'experiments',
    'experiments/example_attack',
    'experiments/example_downstream',
    'experiments/example_downstream_soft',
    'experiments/example_precomputed',
    'experiments/example_precomputed_mix',
    'modules',
    'modules/base_utils',
    'modules/base_utils/model',
    'modules/generate_labels',
    'modules/pytorch_cifar',
    'modules/pytorch_cifar/models',
    'modules/select_flips',
    'modules/train_expert',
    'modules/train_user',
    'schemas',
    'img',
]:
    os.makedirs(d, exist_ok=True)
print('Directories ready.')


## 3. Recreate project files verbatim


### `run_experiment.py`


In [ ]:
%%writefile run_experiment.py
"""
Run experiment based on config.
"""

import sys
import os

import numpy as np
import toml
from collections import OrderedDict

sys.path.insert(0, os.path.abspath(
        os.path.join(os.path.dirname(__file__), 'modules')
    ))

from modules.base_utils import util


experiment_name = sys.argv[1]
args = util.extract_toml(experiment_name)
resolves_to = {}

for module_name, module_config in args.items():
    relative_path = "schemas/" + module_name + ".toml"
    full_path = util.generate_full_path(relative_path)

    # Check if path exists
    if not os.path.exists(full_path):
        print(f"Malformed module! Module {module_name} does not exist!")
        exit()

    schema = toml.load(full_path, _dict=OrderedDict)

    optionals = []
    if 'OPTIONAL' in schema:
        optionals = list(schema['OPTIONAL'].keys())

    # Check if config is well formed
    bad_config = False
    diff_forward = np.setdiff1d(list(schema[module_name].keys()),
                                list(module_config.keys()))
    for item in diff_forward:
        if item not in optionals:
            print(f"Malformed config: {item} exists in schema but not config.")
            bad_config = True

    diff_backward = np.setdiff1d(list(module_config.keys()),
                                 list(schema[module_name].keys()))
    for item in diff_backward:
        if item not in optionals:
            print(f"Malformed config: {item} exists in config but not schema.")
            bad_config = True

    if bad_config:
        exit()

    # Check if module has distinct module name
    if 'INTERNAL' in schema:
        resolves_to[module_name] = schema['INTERNAL']['module_name']

    # Check for slurm and import and run module
    module_file = resolves_to.get(module_name, module_name)

    if os.getenv('SLURM_ARRAY_TASK_ID') is not None:
        slurm_id = int(os.getenv('SLURM_ARRAY_TASK_ID'))
        __import__(f"{module_file}", fromlist=["run_module"]).run_module.run(
            experiment_name, module_name, slurm_id=slurm_id)
    else:
        __import__(f"{module_file}", fromlist=["run_module"]).run_module.run(
            experiment_name, module_name)


### `requirements.txt`


In [ ]:
%%writefile requirements.txt
numpy>=1.20.2,<1.21
pillow>=8.2.0,<8.3
pytorch>=1.11.0,<1.12
torchvision>=0.12.0,<0.13
tqdm>=4.60.0,<4.61
scipy>=1.8.0,<1.9.0
cudatoolkit>=11.3.1,<11.4
pandas>=1.2.0,<1.3.0
python>=3.8.0,<4.0.0
toml>=0.10.0,<0.11.0


### `modules/__init__.py`


In [ ]:
%%writefile modules/__init__.py


### `modules/base_utils/__init__.py`


In [ ]:
%%writefile modules/base_utils/__init__.py


### `modules/base_utils/datasets.py`


In [ ]:
%%writefile modules/base_utils/datasets.py
import random
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset, ConcatDataset, Subset
from torchvision import datasets, transforms
from typing import Callable, Iterable, Tuple
from pathlib import Path
import subprocess


CIFAR_TRANSFORM_NORMALIZE_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_TRANSFORM_NORMALIZE_STD = (0.2023, 0.1994, 0.2010)
CIFAR_TRANSFORM_NORMALIZE = transforms.Normalize(
    CIFAR_TRANSFORM_NORMALIZE_MEAN, CIFAR_TRANSFORM_NORMALIZE_STD
)
CIFAR_TRANSFORM_TRAIN = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        CIFAR_TRANSFORM_NORMALIZE,
    ]
)
CIFAR_TRANSFORM_TEST = transforms.Compose(
    [
        transforms.ToTensor(),
        CIFAR_TRANSFORM_NORMALIZE,
    ]
)

CIFAR_BIG_TRANSFORM_TRAIN = transforms.Compose(
    [
        transforms.Resize(224),
        transforms.RandomCrop(224, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        CIFAR_TRANSFORM_NORMALIZE,
    ]
)

CIFAR_BIG_TRANSFORM_TEST = transforms.Compose(
    [
        transforms.Resize(224),
        transforms.ToTensor(),
        CIFAR_TRANSFORM_NORMALIZE,
    ]
)


CIFAR_100_TRANSFORM_NORMALIZE_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR_100_TRANSFORM_NORMALIZE_STD = (0.2675, 0.2565, 0.2761)
CIFAR_100_TRANSFORM_NORMALIZE = transforms.Normalize(
    CIFAR_100_TRANSFORM_NORMALIZE_MEAN, CIFAR_100_TRANSFORM_NORMALIZE_STD
)
CIFAR_100_TRANSFORM_TRAIN = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        CIFAR_100_TRANSFORM_NORMALIZE,
    ]
)
CIFAR_100_TRANSFORM_TEST = transforms.Compose(
    [
        transforms.ToTensor(),
        CIFAR_100_TRANSFORM_NORMALIZE,
    ]
)


TINY_IMAGENET_TRANSFORM_NORMALIZE_MEAN = (0.485, 0.456, 0.406)
TINY_IMAGENET_TRANSFORM_NORMALIZE_STD = (0.229, 0.224, 0.225)
TINY_IMAGENET_TRANSFORM_NORMALIZE = transforms.Normalize(
    TINY_IMAGENET_TRANSFORM_NORMALIZE_MEAN, TINY_IMAGENET_TRANSFORM_NORMALIZE_STD
)
TINY_IMAGENET_TRANSFORM_TRAIN = transforms.Compose(
    [
        transforms.RandomCrop(64, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        TINY_IMAGENET_TRANSFORM_NORMALIZE,
    ]
)
TINY_IMAGENET_TRANSFORM_TEST = transforms.Compose(
    [
        transforms.ToTensor(),
        TINY_IMAGENET_TRANSFORM_NORMALIZE,
    ]
)

PATH = {
    'cifar': Path("./data/data_cifar10"),
    'cifar_100': Path("./data/data_cifar100"),
    'tiny_imagenet': "/scr/tiny-imagenet-200"
}

TRANSFORM_TRAIN_XY = {
    'cifar': lambda xy: (CIFAR_TRANSFORM_TRAIN(xy[0]), xy[1]),
    'cifar_big': lambda xy: (CIFAR_BIG_TRANSFORM_TRAIN(xy[0]), xy[1]),
    'cifar_100': lambda xy: (CIFAR_100_TRANSFORM_TRAIN(xy[0]), xy[1]),
    'tiny_imagenet': lambda xy: (TINY_IMAGENET_TRANSFORM_TRAIN(xy[0]), xy[1])
}

TRANSFORM_TEST_XY = {
    'cifar': lambda xy: (CIFAR_TRANSFORM_TEST(xy[0]), xy[1]),
    'cifar_big': lambda xy: (CIFAR_BIG_TRANSFORM_TEST(xy[0]), xy[1]),
    'cifar_100': lambda xy: (CIFAR_100_TRANSFORM_TEST(xy[0]), xy[1]),
    'tiny_imagenet': lambda xy: (TINY_IMAGENET_TRANSFORM_TEST(xy[0]), xy[1])
}

N_CLASSES = {
    'cifar': 10,
    'cifar_100': 100,
    'tiny_imagenet': 200
}


class LabelSortedDataset(ConcatDataset):
    def __init__(self, dataset: Dataset):
        self.orig_dataset = dataset
        self.by_label = {}
        for i, (_, y) in enumerate(dataset):
            self.by_label.setdefault(y, []).append(i)
        self.n = len(self.by_label)
        assert set(self.by_label.keys()) == set(range(self.n))
        self.by_label = [Subset(dataset, self.by_label[i])
                         for i in range(self.n)]
        super().__init__(self.by_label)

    def subset(self, labels: Iterable[int]) -> ConcatDataset:
        if isinstance(labels, int):
            labels = [labels]
        return ConcatDataset([self.by_label[i] for i in labels])


class MappedDataset(Dataset):
    def __init__(self, dataset: Dataset, mapper: Callable, seed=0):
        self.dataset = dataset
        self.mapper = mapper
        self.seed = seed

    def __getitem__(self, i: int):
        if hasattr(self.mapper, 'seed'):
            self.mapper.seed(i + self.seed)
        return self.mapper(self.dataset[i])

    def __len__(self):
        return len(self.dataset)


class LabelWrappedDataset(Dataset):
    def __init__(self, dataset: Dataset, labels, train_pct=1.0, include_labels=False):
        self.dataset = dataset
        self.labels = labels
        self.include_labels = include_labels

        if len(labels) < len(dataset):
            self.labels = [y for x, y in self.dataset]
            self.labels[:len(labels)] = labels.tolist()

        assert len(dataset) == len(self.labels)
        

    def __getitem__(self, i: int):
        if self.include_labels:
            return self.dataset[i][0], self.labels[i], self.dataset[i][1]
        return self.dataset[i][0], self.labels[i]

    def __len__(self):
        return len(self.dataset)


class MTTDataset(Dataset):
    def __init__(self, train: Dataset, distill: Dataset, poison_inds, transform, n_classes):
        self.train = train
        self.distill = distill
        self.poison_inds = poison_inds
        self.transform = transform
        self.n_classes = n_classes

    def __getitem__(self, i: int):
        seed = np.random.randint(8)
        random.seed(seed)
        torch.manual_seed(seed)
        train_x, train_y = self.transform(self.train[i])
        train_oh = torch.zeros(self.n_classes)
        train_oh[torch.tensor(train_y)] = 1

        if i >= len(self.distill):
            i = self.poison_inds[i % len(self.distill)]

        random.seed(seed)
        torch.manual_seed(seed)
        distill_x, distill_y = self.transform(self.distill[i])
        distill_oh = torch.zeros(self.n_classes)
        distill_oh[torch.tensor(distill_y)] = 1
        return train_x, train_oh, distill_x, distill_oh, i

    def __len__(self):
        return len(self.train)


class PoisonedDataset(Dataset):
    def __init__(
        self,
        dataset: Dataset,
        poisoner,
        poison_dataset=None,
        *,
        label=None,
        indices=None,
        eps=500,
        seed=1,
        transform=None
    ):
        self.orig_dataset = dataset
        self.label = label
        if not (indices or eps):
            raise ValueError()

        if not indices:
            if label is not None:
                clean_inds = [i for i, (x, y) in enumerate(dataset)
                              if y == label]
            else:
                clean_inds = range(len(dataset))

            rng = np.random.RandomState(seed)
            indices = rng.choice(clean_inds, eps, replace=False)

        self.indices = indices
        self.poison_dataset = MappedDataset(Subset(poison_dataset or dataset, indices),
                                            poisoner,
                                            seed=seed)

        if transform:
            self.poison_dataset = MappedDataset(self.poison_dataset, transform)

        clean_indices = list(set(range(len(dataset))).difference(indices))
        self.clean_dataset = Subset(dataset, clean_indices)

        if transform:
            self.clean_dataset = MappedDataset(self.clean_dataset, transform)

        self.dataset = ConcatDataset([self.clean_dataset, self.poison_dataset])

    def __getitem__(self, i: int):
        return self.dataset[i]

    def __len__(self):
        return len(self.dataset)


class Poisoner(object):
    def poison(self, x: Image.Image) -> Image.Image:
        raise NotImplementedError()

    def __call__(self, x: Image.Image) -> Image.Image:
        return self.poison(x)


class PixelPoisoner(Poisoner):
    def __init__(
        self,
        *,
        method="pixel",
        pos: Tuple[int, int] = (11, 16),
        col: Tuple = (101, 0, 25)
    ):
        self.method = method
        self.pos = pos
        self.col = col

    def poison(self, x: Image.Image) -> Image.Image:
        ret_x = x.copy()
        pos, col = self.pos, self.col

        if self.method == "pixel":
            ret_x.putpixel(pos, col)
        elif self.method == "pattern":
            ret_x.putpixel(pos, col)
            ret_x.putpixel((pos[0] - 1, pos[1] - 1), col)
            ret_x.putpixel((pos[0] - 1, pos[1] + 1), col)
            ret_x.putpixel((pos[0] + 1, pos[1] - 1), col)
            ret_x.putpixel((pos[0] + 1, pos[1] + 1), col)
        elif self.method == "ell":
            ret_x.putpixel(pos, col)
            ret_x.putpixel((pos[0] + 1, pos[1]), col)
            ret_x.putpixel((pos[0], pos[1] + 1), col)

        return ret_x


class TurnerPoisoner(Poisoner):
    def __init__(
        self,
        *,
        method="bottom-right"
    ):
        self.method = method
        self.trigger_mask = [
            ((-1, -1), 1),
            ((-1, -2), -1),
            ((-1, -3), 1),
            ((-2, -1), -1),
            ((-2, -2), 1),
            ((-2, -3), -1),
            ((-3, -1), 1),
            ((-3, -2), -1),
            ((-3, -3), -1)
        ]

    def poison(self, x: Image.Image) -> Image.Image:
        ret_x = x.copy()
        px = ret_x.load()

        for (x, y), sign in self.trigger_mask:
            shift = int(sign * 255)
            r, g, b = px[x, y]
            shifted = (r + shift, g + shift, b + shift)
            px[x, y] = shifted
            if self.method == "all-corners":
                px[-x - 1, y] = px[x, -y - 1] = px[-x - 1, -y - 1] = shifted

        return ret_x


class StripePoisoner(Poisoner):
    def __init__(self, *, horizontal=True, strength=6, freq=16):
        self.horizontal = horizontal
        self.strength = strength
        self.freq = freq

    def poison(self, x: Image.Image) -> Image.Image:
        arr = np.asarray(x)
        (w, h, d) = arr.shape
        assert w == h  # have not tested w != h
        mask = np.full(
            (d, w, h), np.sin(np.linspace(0, self.freq * np.pi, h))
        ).swapaxes(0, 2)
        if self.horizontal:
            mask = mask.swapaxes(0, 1)
        mix = np.asarray(x) + self.strength * mask
        return Image.fromarray(np.uint8(mix.clip(0, 255)))


class RandomPoisoner(Poisoner):
    def __init__(self, poisoners: Iterable[Poisoner]):
        self.poisoners = poisoners
        self.rng = np.random.RandomState()

    def poison(self, x):
        poisoner = self.rng.choice(self.poisoners)
        return poisoner.poison(x)

    def seed(self, i):
        self.rng.seed(i)


class LabelPoisoner(Poisoner):
    def __init__(self, poisoner: Poisoner, target_label: int):
        self.poisoner = poisoner
        self.target_label = target_label

    def poison(self, xy):
        x, _ = xy
        return self.poisoner(x), self.target_label

    def seed(self, i):
        if hasattr(self.poisoner, 'seed'):
            self.poisoner.seed(i)


def load_dataset(dataset_flag, train=True):
    path = PATH[dataset_flag]
    if dataset_flag == 'cifar':
        return load_cifar_dataset(path, train)
    if dataset_flag == 'cifar_100':
        return load_cifar_100_dataset(path, train)
    elif dataset_flag == 'tiny_imagenet':
        return load_tiny_imagenet_dataset(path, train)
    else:
        raise NotImplementedError(f"Dataset {dataset_flag} is not supported.")


def load_cifar_dataset(path, train=True):
    dataset = datasets.CIFAR10(root=str(path),
                               train=train,
                               download=True)
    return dataset


def load_cifar_100_dataset(path, train=True, coarse=True):
    dataset = datasets.CIFAR100(root=str(path),
                                train=train,
                                download=True)

    if coarse:
        coarse_labels = np.array([ 4,  1, 14,  8,  0,  6,  7,  7, 18,  3,
                                   3, 14,  9, 18,  7, 11,  3,  9,  7, 11,
                                   6, 11,  5, 10,  7,  6, 13, 15,  3, 15,
                                   0, 11,  1, 10, 12, 14, 16,  9, 11,  5,
                                   5, 19,  8,  8, 15, 13, 14, 17, 18, 10,
                                   16, 4, 17,  4,  2,  0, 17,  4, 18, 17,
                                   10, 3,  2, 12, 12, 16, 12,  1,  9, 19,
                                   2, 10,  0,  1, 16, 12,  9, 13, 15, 13,
                                  16, 19,  2,  4,  6, 19,  5,  5,  8, 19,
                                  18,  1,  2, 15,  6,  0, 17,  8, 14, 13])
        dataset.targets = coarse_labels[dataset.targets]
        dataset.classes = range(coarse_labels.max()+1)
    return dataset


def load_tiny_imagenet_dataset(path, train=True):
    if not Path(PATH["tiny_imagenet"]).is_dir():
        command = ["./modules/base_utils/tiny_imagenet_setup.sh"]
        print("Downloading Tiny ImageNet Dataset...")
        process = subprocess.Popen(command, shell=True, stdout=subprocess.DEVNULL)
        process.wait()
    path = path + ("/train" if train else "/val/images")
    dataset = datasets.ImageFolder(path)
    return dataset

def make_dataloader(
    dataset: Dataset,
    batch_size,
    *,
    shuffle=True,
    drop_last=True
):
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=4,
        pin_memory=True,
        drop_last=drop_last,
    )
    return dataloader


def pick_poisoner(poisoner_flag, dataset_flag, target_label):
    if dataset_flag == "cifar" or dataset_flag == "cifar_100":
        x_poisoner = pick_cifar_poisoner(poisoner_flag)
    elif dataset_flag == "tiny_imagenet":
        x_poisoner = pick_tiny_imagenet_poisoner(poisoner_flag)
    else:
        raise NotImplementedError()

    x_label_poisoner = LabelPoisoner(x_poisoner, target_label=target_label)

    return x_label_poisoner


def pick_cifar_poisoner(poisoner_flag):
    if poisoner_flag == "1xp":
        x_poisoner = PixelPoisoner()

    elif poisoner_flag == "2xp":
        x_poisoner = RandomPoisoner(
            [
                PixelPoisoner(),
                PixelPoisoner(pos=(5, 27), col=(101, 123, 121)),
            ]
        )

    elif poisoner_flag == "3xp":
        x_poisoner = RandomPoisoner(
            [
                PixelPoisoner(),
                PixelPoisoner(pos=(5, 27), col=(101, 123, 121)),
                PixelPoisoner(pos=(30, 7), col=(0, 36, 54)),
            ]
        )

    elif poisoner_flag == "1xs":
        x_poisoner = StripePoisoner(strength=6, freq=16)

    elif poisoner_flag == "2xs":
        x_poisoner = RandomPoisoner(
            [
                StripePoisoner(strength=6, freq=16),
                StripePoisoner(strength=6, freq=16, horizontal=False),
            ]
        )

    elif poisoner_flag == "1xl":
        x_poisoner = TurnerPoisoner()

    elif poisoner_flag == "4xl":
        x_poisoner = TurnerPoisoner(method="all-corners")

    else:
        raise NotImplementedError()

    return x_poisoner


def pick_tiny_imagenet_poisoner(poisoner_flag):
    if poisoner_flag == "1xp":
        x_poisoner = PixelPoisoner(pos=(22, 32), col=(101, 0, 25))

    elif poisoner_flag == "2xp":
        x_poisoner = RandomPoisoner(
            [
                PixelPoisoner(pos=(22, 32), col=(101, 0, 25)),
                PixelPoisoner(pos=(10, 54), col=(101, 123, 121)),
            ]
        )

    elif poisoner_flag == "3xp":
        x_poisoner = RandomPoisoner(
            [
                PixelPoisoner(pos=(22, 32), col=(101, 0, 25)),
                PixelPoisoner(pos=(10, 54), col=(101, 123, 121)),
                PixelPoisoner(pos=(60, 14), col=(0, 36, 54)),
            ]
        )

    elif poisoner_flag == "1xs":
        x_poisoner = StripePoisoner(strength=6, freq=16)

    elif poisoner_flag == "2xs":
        x_poisoner = RandomPoisoner(
            [
                StripePoisoner(strength=6, freq=16),
                StripePoisoner(strength=6, freq=16, horizontal=False),
            ]
        )

    elif poisoner_flag == "1xl":
        x_poisoner = TurnerPoisoner()

    elif poisoner_flag == "4xl":
        x_poisoner = TurnerPoisoner(method="all-corners")

    else:
        raise NotImplementedError()

    return x_poisoner


def get_matching_datasets(
    dataset_flag,
    poisoner,
    label,
    seed=1,
    train_pct=1.0,
    big=False
):
    train_transform = TRANSFORM_TRAIN_XY[dataset_flag + ('_big' if big else '')]
    test_transform = TRANSFORM_TEST_XY[dataset_flag + ('_big' if big else '')]

    train_data = load_dataset(dataset_flag, train=True)
    test_data = load_dataset(dataset_flag, train=False)

    n_classes = len(train_data.classes)
    train_labels = np.array([y for _, y in train_data])

    train_labels = train_labels[:int(len(train_labels) * train_pct)]

    n_poisons_train = int((len(train_data) // n_classes) * train_pct)
    n_poisons_test = len(test_data) // n_classes

    if label == -1:
        poison_inds = np.where(train_labels != poisoner.target_label)[0][-n_poisons_train:]
    else:
        poison_inds = np.where(train_labels == label)[0][-n_poisons_train:]

    mtt_distill_dataset = distill_dataset = Subset(train_data, np.arange(len(train_data)))
    poison_dataset = MappedDataset(Subset(train_data, poison_inds),
                                   poisoner,
                                   seed=seed)

    train_dataset = Subset(train_data, np.arange(int(len(train_data) * train_pct)))
    dataset_list = [train_dataset, poison_dataset]
    if dataset_flag == 'tiny_imagenet':   # Oversample poisons for expert training
        dataset_list.extend([poison_dataset] * 9)
    train_dataset = ConcatDataset(dataset_list)

    if train_pct < 1.0:
        mtt_distill_dataset = Subset(distill_dataset, np.arange(int(len(distill_dataset) * train_pct)))

    mtt_dataset = MTTDataset(train_dataset, mtt_distill_dataset, poison_inds,
                             train_transform, n_classes)

    distill_dataset = MappedDataset(distill_dataset, train_transform)
    train_dataset = MappedDataset(train_dataset, train_transform)
    test_dataset = MappedDataset(test_data, test_transform)
    poison_test_dataset = PoisonedDataset(
        test_data,
        poisoner,
        eps=n_poisons_test,
        label=label if label != -1 else None,
        transform=test_transform,
    )

    return train_dataset, distill_dataset, test_dataset, poison_test_dataset, mtt_dataset


def construct_user_dataset(distill_dataset, labels, mask=None, target_label=None, include_labels=False):
    dataset = LabelWrappedDataset(distill_dataset, labels, include_labels)
    return dataset

def get_n_classes(dataset_flag):
    return N_CLASSES[dataset_flag]


### `modules/base_utils/util.py`


In [ ]:
%%writefile modules/base_utils/util.py
import os
import numpy as np
import torch
import tqdm
from functools import partial
from torch import optim
from torch.utils.data import DataLoader, Dataset
from typing import Iterable, Union
from modules.base_utils.model.model import SequentialImageNetwork,\
                                   SequentialImageNetworkMod
import torch.backends.cudnn as cudnn
import toml
from collections import OrderedDict

from modules.base_utils.datasets import make_dataloader

if torch.cuda.is_available():
    cudnn.benchmark = True

default_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

DEFAULT_SGD_BATCH_SIZE = 256
DEFAULT_SGD_EPOCHS = 200
DEFAULT_SGD_KWARGS = {
    'lr': 0.1,
    'momentum': 0.9,
    'nesterov': True,
    'weight_decay': 2e-4
}
DEFAULT_SGD_SCHED_KWARGS = {
    'milestones': [75, 125],
    'gamma': 0.1
}


DEFAULT_ADAM_BATCH_SIZE = 256
DEFAULT_ADAM_EPOCHS = 200
DEFAULT_ADAM_KWARGS = {
    'lr': 0.001,
    'betas': (0.9, 0.999),
    'weight_decay': 1e-4
}
DEFAULT_ADAM_SCHED_KWARGS = {
    'milestones': [125],
    'gamma': 0.1
}

BIG_IMS_MODELS = ['vgg', 'vgg-pretrain', 'vit-pretrain']


def generate_full_path(path):
    return os.path.join(os.getcwd(), path)


def slurmify_path(path, slurm_id):
    if path is None:
        return path
    return path if slurm_id is None else path.format(slurm_id)


def extract_toml(experiment_name, module_name=None):
    relative_path = "experiments/" + experiment_name + "/config.toml"
    full_path = generate_full_path(relative_path)
    assert os.path.exists(full_path)

    exp_toml = toml.load(full_path, _dict=OrderedDict)
    if module_name is not None:
        return exp_toml[module_name]
    return exp_toml
 

def load_model(model_flag, num_classes=10):
    if num_classes != 10 and model_flag not in ['r32p', 'r18', 'r18-tin']:
        raise NotImplementedError

    if model_flag == "r32p":
        import modules.base_utils.model.resnet as resnet

        return SequentialImageNetworkMod(resnet.resnet32(num_classes)).cuda()
    elif model_flag == "r18":
        from pytorch_cifar.models import ResNet, BasicBlock        
        return SequentialImageNetwork(ResNet(BasicBlock,
                                             [2, 2, 2, 2],
                                             num_classes)).cuda()
    elif model_flag == "r18-tin":
        from pytorch_cifar.models import ResNet, BasicBlock

        model = SequentialImageNetwork(ResNet(BasicBlock,
                                             [2, 2, 2, 2],
                                             num_classes))
        model[13] = torch.nn.Linear(2048, 200)

        return model.cuda()
    elif model_flag == "vgg":
        from torchvision.models import vgg19_bn
        
        return vgg19_bn(num_classes=num_classes).cuda()
    elif model_flag == "vgg-pretrain":
        from torchvision.models import vgg19_bn, VGG19_BN_Weights
        model = vgg19_bn(weights=VGG19_BN_Weights.DEFAULT)
        model.classifier[6] = torch.nn.Linear(4096, num_classes)
        for name, param in model.named_parameters():
            if 'classifier' not in name:
                param.requires_grad=False

        return model.cuda()
    elif model_flag == "vit-pretrain":
        from torchvision.models import vit_b_16, ViT_B_16_Weights
        model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        model.heads.head = torch.nn.Linear(768, num_classes)
        for name, param in model.named_parameters():
            if 'head' not in name:
                param.requires_grad=False
        return model.cuda()
    else:
        raise NotImplementedError


def make_pbar(*args, **kwargs):
    pbar_constructor = (
        partial(tqdm.tqdm, dynamic_ncols=True)
    )
    return pbar_constructor(*args, **kwargs)


def get_module_device(module: torch.nn.Module, check=True):
    if check:
        assert len(set(param.device for param in module.parameters())) == 1
    return next(module.parameters()).device


def either_dataloader_dataset_to_both(
    data: Union[DataLoader, Dataset], *, batch_size=None, eval=False, **kwargs
):
    if isinstance(data, DataLoader):
        dataloader = data
        dataset = data.dataset
    elif isinstance(data, Dataset):
        dataset = data
        dl_kwargs = {}

        if eval:
            dl_kwargs.update(dict(batch_size=256, shuffle=False,
                                  drop_last=False))
        else:
            dl_kwargs.update(dict(batch_size=128, shuffle=True))

        if batch_size is not None:
            dl_kwargs["batch_size"] = batch_size

        dl_kwargs.update(kwargs)
        dataloader = make_dataloader(data, **dl_kwargs)
    else:
        raise NotImplementedError()
    return dataloader, dataset


clf_loss = torch.nn.CrossEntropyLoss()
total_mse_distance = torch.nn.MSELoss(reduction="sum")
softmax = torch.nn.Softmax(dim=1)


def clf_correct(y_pred: torch.Tensor, y: torch.Tensor):
    if len(y.shape) > 1 and y.shape[1] > 1:
        y = torch.argmax(y, dim=1)
    y_hat = y_pred.data.max(1)[1]
    correct = (y_hat == y).long().cpu().sum()
    return correct


def clf_eval(model: torch.nn.Module, data: Union[DataLoader, Dataset]):
    device = get_module_device(model)
    dataloader, _ = either_dataloader_dataset_to_both(data, eval=True)
    total_correct, total_loss = 0.0, 0.0
    with torch.no_grad():
        model.eval()
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            y_pred = model(x)
            loss = clf_loss(y_pred, y)
            correct = clf_correct(y_pred, y)

            total_correct += correct.item()
            total_loss += loss.item()

    n = len(dataloader.dataset)
    total_correct /= n
    total_loss /= n
    return total_correct, total_loss


def get_mean_lr(opt: optim.Optimizer):
    return np.mean([group["lr"] for group in opt.param_groups])


def mini_train(
    *,
    model: torch.nn.Module,
    train_data: Union[DataLoader, Dataset],
    test_data: Union[Union[DataLoader, Dataset],
                     Iterable[Union[DataLoader, Dataset]]] = None,
    batch_size=32,
    opt: optim.Optimizer,
    scheduler,
    epochs: int,
    shuffle=True,
    callback=None,
    record=False
):
    device = get_module_device(model)
    dataloader, _ = either_dataloader_dataset_to_both(train_data,
                                                      batch_size=batch_size,
                                                      shuffle=shuffle)
    n = len(dataloader.dataset)
    total_examples = epochs * n

    if test_data:
        num_sets = 1
        if isinstance(test_data, Iterable):
            num_sets = len(test_data)
        else:
            test_data = [test_data]
        acc_loss = [[] for _ in range(num_sets)]

    with make_pbar(total=total_examples) as pbar:
        for epoch in range(1, epochs + 1):
            train_epoch_loss, train_epoch_correct = 0, 0
            model.train()
            for i, (x, y) in enumerate(dataloader):
                x, y = x.to(device), y.to(device)
                minibatch_size = len(x)
                model.zero_grad()
                y_pred = model(x)
                loss = clf_loss(y_pred, y)
                correct = clf_correct(y_pred, y)
                loss.backward()
                opt.step()
                train_epoch_correct += int(correct.item())
                train_epoch_loss += float(loss.item())
                pbar.update(minibatch_size)
                if callback is not None:
                    callback(model, opt, epoch, i)

            lr = get_mean_lr(opt)
            if scheduler:
                scheduler.step()

            pbar_postfix = {
                "acc": "%.2f" % (train_epoch_correct / n * 100),
                "loss": "%.4g" % (train_epoch_loss / n),
                "lr": "%.3g" % lr,
            }
            if test_data:
                for i, dataset in enumerate(test_data):
                    acc, loss = clf_eval(model, dataset)
                    pbar_postfix.update(
                        {
                            "acc" + str(i): "%.2f" % (acc * 100),
                            # "loss" + str(i): "%.4g" % loss,
                        }
                    )
                    if record:
                        acc_loss[i].append((acc, loss))
            pbar.set_postfix(**pbar_postfix)

    if record:
        return model, *acc_loss
    return model


def get_train_info(
    params,
    train_flag,
    batch_size=None,
    epochs=None,
    optim_kwargs={},
    scheduler_kwargs={}
):
    if train_flag == "sgd":
        batch_size = batch_size or DEFAULT_SGD_BATCH_SIZE
        epochs = epochs or DEFAULT_SGD_EPOCHS
        kwargs = {**DEFAULT_SGD_KWARGS, **optim_kwargs}
        sched_kwargs = {**DEFAULT_SGD_SCHED_KWARGS, **scheduler_kwargs}
        opt = optim.SGD(params, **kwargs)
        lr_scheduler = optim.lr_scheduler.MultiStepLR(opt, **sched_kwargs)
    elif train_flag == "adam":
        batch_size = batch_size or DEFAULT_ADAM_BATCH_SIZE
        epochs = epochs or DEFAULT_ADAM_EPOCHS
        kwargs = {**DEFAULT_ADAM_KWARGS, **optim_kwargs}
        sched_kwargs = {**DEFAULT_ADAM_SCHED_KWARGS, **scheduler_kwargs}
        opt = optim.Adam(params, **kwargs)
        lr_scheduler = optim.lr_scheduler.MultiStepLR(opt, **sched_kwargs)
    else:
        raise NotImplementedError

    return batch_size, epochs, opt, lr_scheduler


def get_mtt_attack_info(
    expert_params,
    labels,
    expert_kwargs,
    labels_kwargs,
    train_flag='sgd',
    batch_size=None,
    epochs=None
):
    if train_flag == "sgd":
        batch_size = batch_size or DEFAULT_SGD_BATCH_SIZE
        epochs = epochs or DEFAULT_SGD_EPOCHS
        opt_expert = optim.SGD(expert_params, **expert_kwargs)
        opt_labels = optim.SGD([labels], **labels_kwargs)
    else:
        raise NotImplementedError

    assert len(opt_expert.state_dict()['param_groups']) == 1
    return batch_size, epochs, opt_expert, opt_labels


def needs_big_ims(model_flag):
    return model_flag in BIG_IMS_MODELS


### `modules/base_utils/tiny_imagenet_fix_val.py`


In [ ]:
%%writefile modules/base_utils/tiny_imagenet_fix_val.py
import os

DATA_DIR = 'data/tiny-imagenet-200/'
VALID_DIR = DATA_DIR + 'val'

# Create separate validation subfolders for the validation images based on
# their labels indicated in the val_annotations txt file
val_img_dir = os.path.join(VALID_DIR, 'images')
fp = open(os.path.join(VALID_DIR, 'val_annotations.txt'), 'r')
data = fp.readlines()

# Create dictionary to store img filename (word 0) and corresponding
# label (word 1) for every line in the txt file (as key value pair)
val_img_dict = {}
for line in data:
    words = line.split('\t')
    val_img_dict[words[0]] = words[1]
fp.close()

# Create subfolders (if not present) for validation images based on label ,
# and move images into the respective folders
for img, folder in val_img_dict.items():
    newpath = (os.path.join(val_img_dir, folder))
    if not os.path.exists(newpath):
        os.makedirs(newpath)
    if os.path.exists(os.path.join(val_img_dir, img)):
        os.rename(os.path.join(val_img_dir, img), os.path.join(newpath, img))

# Save class names (for corresponding labels) as dict from words.txt file
class_to_name_dict = dict()
fp = open(os.path.join(DATA_DIR, 'words.txt'), 'r')
data = fp.readlines()
for line in data:
    words = line.strip('\n').split('\t')
    class_to_name_dict[words[0]] = words[1].split(',')[0]
fp.close()


### `modules/base_utils/tiny_imagenet_setup.sh`


In [ ]:
%%writefile modules/base_utils/tiny_imagenet_setup.sh
wget -nc http://cs231n.stanford.edu/tiny-imagenet-200.zip
unzip tiny-imagenet-200.zip -d data/
python modules/base_utils/tiny_imagenet_fix_val.py

### `modules/base_utils/model/model.py`


In [ ]:
%%writefile modules/base_utils/model/model.py
import torch.nn as nn
import torch.nn.functional as F
import modules.base_utils.model.resnet as resnet_mod
import modules.pytorch_cifar.models.resnet as resnet
import collections


class BasicBlockNoReLU(nn.Module):
    expansion = 1

    def __init__(self, module):
        super().__init__()
        self.conv1 = module.conv1
        self.bn1 = module.bn1
        self.conv2 = module.conv2
        self.bn2 = module.bn2
        self.shortcut = module.shortcut

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return out


class SequentialImageNetwork(nn.Sequential):
    def __init__(self, net=resnet.ResNet18()):
        if isinstance(net, collections.OrderedDict):
            super().__init__(net)
            return

        self.net_holder = (net,)
        i = 1
        layers = []
        while hasattr(net, name := f"layer{i}"):
            layers.extend(list(getattr(net, name)))
            i += 1

        layers2 = []
        for layer in layers:
            if isinstance(layer, resnet.BasicBlock):
                layers2.append(BasicBlockNoReLU(layer))
                layers2.append(nn.ReLU())
            else:
                layers2.append(layer)

        super().__init__(
            net.conv1,
            net.bn1,
            nn.ReLU(),
            *layers2,
            nn.AvgPool2d(8 if net.in_planes == 64 else 4),
            nn.Flatten(),
            net.linear,
        )

    @property
    def net(self):
        return self.net_holder[0]


class BasicBlockSplitter(nn.Module):
    def __init__(self, block: resnet.BasicBlock, step="add"):
        super().__init__()
        self.block = block
        self.step = step

    def forward(self, x):
        if self.step == "identity":
            return x
        shortcut = self.block.shortcut(x)
        if self.step == "shortcut":
            return shortcut
        x = self.block.conv1(x)
        if self.step == "conv1":
            return x
        x = self.block.bn1(x)
        if self.step == "bn1":
            return x
        x = F.relu(x)
        if self.step == "relu1":
            return x
        x = self.block.conv2(x)
        if self.step == "conv2":
            return x
        x = self.block.bn2(x)
        if self.step == "bn2":
            return x
        x += shortcut
        if self.step == "add":
            return x
        x = F.relu(x)
        if self.step == "relu2":
            return x
        return x


class SequentialImageNetworkMod(nn.Sequential):
    def __init__(self, net=resnet_mod.resnet32()):
        if isinstance(net, collections.OrderedDict):
            super().__init__(net)
            return

        self.net_holder = (net,)
        i = 1
        layers = []
        while hasattr(net, name := f"layer{i}"):
            layers.extend(list(getattr(net, name)))
            i += 1

        super().__init__(
            net.conv1,
            *layers,
            net.final_bn,
            nn.LeakyReLU(0.1),
            nn.AvgPool2d(8),
            nn.Flatten(),
            net.linear,
        )

    @property
    def net(self):
        return self.net_holder[0]


### `modules/base_utils/model/resnet.py`


In [ ]:
%%writefile modules/base_utils/model/resnet.py
'''
Properly implemented ResNet-s for CIFAR10 as described in paper [1].

The implementation and structure of this file is hugely influenced by [2]
which is implemented for ImageNet and doesn't have option A for identity.
Moreover, most of the implementations on the web is copy-paste from
torchvision's resnet and has wrong number of params.

Proper ResNet-s for CIFAR10 (for fair comparision and etc.) has following
number of layers and parameters:

name      | layers | params
ResNet20  |    20  | 0.27M
ResNet32  |    32  | 0.46M
ResNet44  |    44  | 0.66M
ResNet56  |    56  | 0.85M
ResNet110 |   110  |  1.7M
ResNet1202|  1202  | 19.4m

which this implementation indeed has.

Reference:
[1] Kaiming He, Xiangyu Zhang, Shaoqing Ren, Jian Sun
    Deep Residual Learning for Image Recognition. arXiv:1512.03385
[2] https://github.com/pytorch/vision/blob/master/torchvision/models/resnet.py

If you use this implementation in you work, please don't forget to mention the
author, Yerlan Idelbayev.
'''

# Modified by Jonathan Hayase to match the experimental setup
# used in https://github.com/MadryLab/backdoor_data_poisoning

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init

from torch.autograd import Variable

__all__ = ['ResNet', 'resnet20', 'resnet32', 'resnet44', 'resnet56', 'resnet110', 'resnet1202']

def _weights_init(m):
    classname = m.__class__.__name__
    #print(classname)
    if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
        init.kaiming_normal_(m.weight)

class LambdaLayer(nn.Module):
    def __init__(self, lambd):
        super(LambdaLayer, self).__init__()
        self.lambd = lambd

    def forward(self, x):
        return self.lambd(x)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, activate_before_residual=False, option='A'):
        super(BasicBlock, self).__init__()
        self.activate_before_residual = activate_before_residual

        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            if option == 'A':
                """
                For CIFAR10 ResNet paper uses option A.
                """
                self.shortcut = nn.Sequential(
                    nn.AvgPool2d(stride),
                    LambdaLayer(lambda x:
                    F.pad(x, (0, 0, 0, 0, planes//4, planes//4), "constant", 0))
                )
                # self.shortcut = LambdaLayer(lambda x:
                #                             F.pad(x[:, :, ::2, ::2], (0, 0, 0, 0, planes//4, planes//4), "constant", 0))
            elif option == 'B':
                self.shortcut = nn.Sequential(
                    nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                    nn.BatchNorm2d(self.expansion * planes)
                )

    def forward(self, x):
        if self.activate_before_residual:
            x = self.bn1(x)
            x = F.leaky_relu(x, 0.1)
            out = x
        else:
            out = self.bn1(x)
            out = F.leaky_relu(out, 0.1)

        out = F.leaky_relu(self.bn2(self.conv1(out)), 0.1)
        out = self.conv2(out)
        out += self.shortcut(x)
        return out


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 16

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)
        self.final_bn = nn.BatchNorm2d(64)
        self.linear = nn.Linear(64, num_classes)

        self.apply(_weights_init)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for i, stride in enumerate(strides):
            activate_before_residual = self.in_planes == 16 and i == 0
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion

        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.final_bn(out)
        out = F.leaky_relu(out, 0.1)
        out = F.avg_pool2d(out, out.size()[3])
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def resnet20(num_classes=10):
    return ResNet(BasicBlock, [3, 3, 3], num_classes)


def resnet32(num_classes=10):
    return ResNet(BasicBlock, [5, 5, 5], num_classes)


def resnet44(num_classes=10):
    return ResNet(BasicBlock, [7, 7, 7], num_classes)


def resnet56(num_classes=10):
    return ResNet(BasicBlock, [9, 9, 9], num_classes)


def resnet110(num_classes=10):
    return ResNet(BasicBlock, [18, 18, 18], num_classes)


def resnet1202(num_classes=10):
    return ResNet(BasicBlock, [200, 200, 200], num_classes)


def test(net):
    import numpy as np
    total_params = 0

    for x in filter(lambda p: p.requires_grad, net.parameters()):
        total_params += np.prod(x.data.numpy().shape)
    print("Total number of params", total_params)
    print("Total layers", len(list(filter(lambda p: p.requires_grad and len(p.data.size())>1, net.parameters()))))


if __name__ == "__main__":
    for net_name in __all__:
        if net_name.startswith('resnet'):
            print(net_name)
            test(globals()[net_name]())
            print()


### `modules/generate_labels/run_module.py`


In [ ]:
%%writefile modules/generate_labels/run_module.py
"""
Optimizes logit labels given expert trajectories using trajectory matching.
"""

from pathlib import Path
import sys

import torch
import numpy as np

from modules.base_utils.datasets import get_matching_datasets, pick_poisoner, get_n_classes
from modules.base_utils.util import extract_toml, get_module_device, get_mtt_attack_info,\
                                    load_model, either_dataloader_dataset_to_both, make_pbar,\
                                    needs_big_ims, slurmify_path, clf_loss, softmax,\
                                    total_mse_distance
from modules.generate_labels.utils import coalesce_attack_config, extract_experts,\
                                          extract_labels, sgd_step


def run(experiment_name, module_name, **kwargs):
    """
    Optimizes and saves poisoned logit labels.

    :param experiment_name: Name of the experiment in configuration.
    :param module_name: Name of the module in configuration.
    :param kwargs: Additional arguments (such as slurm id).
    """

    slurm_id = kwargs.get('slurm_id', None)

    args = extract_toml(experiment_name, module_name)

    input_pths = args["input_pths"]
    opt_pths = args["opt_pths"]
    expert_model_flag = args["expert_model"]
    dataset_flag = args["dataset"]
    poisoner_flag = args["poisoner"]
    clean_label = args["source_label"]
    target_label = args["target_label"]
    lam = args.get("lambda", 0.0)
    train_pct = args.get("train_pct", 1.0)
    batch_size = args.get("batch_size", None)
    epochs = args.get("epochs", None)
    expert_config = args.get('expert_config', {})
    config = coalesce_attack_config(args.get("attack_config", {}))

    output_dir = slurmify_path(args["output_dir"], slurm_id)
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Build datasets and initialize labels
    print("Building datasets...")
    poisoner = pick_poisoner(poisoner_flag,
                             dataset_flag,
                             target_label)

    big_ims = needs_big_ims(expert_model_flag)
    _, _, _, _, mtt_dataset =\
        get_matching_datasets(dataset_flag, poisoner, clean_label, train_pct=train_pct, big=big_ims)
    
    labels = extract_labels(mtt_dataset.distill, config['one_hot_temp'], n_classes)
    labels_init = torch.stack(extract_labels(mtt_dataset.distill, 1, n_classes))
    labels_syn = torch.stack(labels).requires_grad_(True)

    # Load expert trajectories
    print("Loading expert trajectories...")
    expert_starts, expert_opt_starts = extract_experts(
        expert_config,
        input_pths,
        config['iterations'],
        expert_opt_path=opt_pths
    )

    # Optimize labels
    print("Training...")
    n_classes = get_n_classes(dataset_flag)

    student_model = load_model(expert_model_flag, n_classes)
    expert_model = load_model(expert_model_flag, n_classes)

    device = get_module_device(student_model)

    batch_size, epochs, optimizer_expert, optimizer_labels = get_mtt_attack_info(
        expert_model.parameters(),
        labels_syn,
        config['expert_kwargs'],
        config['labels_kwargs'],
        batch_size=batch_size,
        epochs=epochs
    )

    mtt_dataloader, _ = either_dataloader_dataset_to_both(mtt_dataset,
                                                          batch_size=batch_size)

    losses = []
    with make_pbar(total=config['iterations'] * len(mtt_dataset)) as pbar:
        for i in range(config['iterations']):
            for x_t, y_t, x_d, y_true, idx in mtt_dataloader:
                # Prepare data
                y_d = labels_syn[idx]
                x_t, y_t, x_d, y_d = x_t.to(device), y_t.to(device), x_d.to(device), y_d.to(device)

                # Load parameters
                checkpoint = torch.load(expert_starts[i])
                expert_model.load_state_dict(checkpoint)
                student_model.load_state_dict({k: v.clone() for k, v in checkpoint.items()})
                expert_start = [v.clone() for v in expert_model.parameters()]

                optimizer_expert.load_state_dict(torch.load(expert_opt_starts[i]))
                state_dict = torch.load(expert_opt_starts[i])

                # Take a single expert / poison step
                expert_model.train()
                expert_model.zero_grad()
                loss = clf_loss(expert_model(x_t), y_t)
                loss.backward()
                optimizer_expert.step()
                expert_model.eval()

                # Train a single student step
                student_model.train()
                student_model.zero_grad()

                loss = clf_loss(student_model(x_d), softmax(y_d))
                grads = torch.autograd.grad(loss, student_model.parameters(), create_graph=True)

                # Calculate loss
                param_loss = torch.tensor(0.0).to(device)
                param_dist = torch.tensor(0.0).to(device)

                for initial, student, expert, grad, state in zip(expert_start,
                                                                 student_model.parameters(),
                                                                 expert_model.parameters(),
                                                                 grads,
                                                                 state_dict['state'].values()):
                    student_update = sgd_step(student, grad, state, state_dict['param_groups'][0])

                    param_loss += total_mse_distance(student_update, expert)
                    param_dist += total_mse_distance(initial, expert)

                # Add Regularization and calculate loss
                reg_term = lam * torch.linalg.vector_norm(softmax(labels_syn) - labels_init, ord=1, axis=1).mean()
                grand_loss = (param_loss / param_dist) + reg_term
                g_loss = grand_loss.item()

                # Optimize labels and learning rate
                optimizer_labels.zero_grad()
                grand_loss.backward()
                optimizer_labels.step()

                # Record training information
                losses.append(g_loss)
                pbar.update(batch_size)
                pbar_postfix = {
                    'g_loss': "%.4g" % np.mean(losses[-20:]),
                    'reg_term':"%.4g" % reg_term,
                }
                pbar.set_postfix(**pbar_postfix)

    # Save results
    print("Saving results...")
    y_true = torch.stack([mtt_dataset[i][3].detach() for i in range(len(mtt_dataset.distill))])
    np.save(output_dir + "labels.npy", labels_syn.detach().numpy())
    np.save(output_dir + "true.npy", y_true)
    np.save(output_dir + "losses.npy", losses)

if __name__ == "__main__":
    experiment_name, module_name = sys.argv[1], sys.argv[2]
    run(experiment_name, module_name)


### `modules/generate_labels/utils.py`


In [ ]:
%%writefile modules/generate_labels/utils.py
from modules.base_utils.util import DEFAULT_SGD_KWARGS
import numpy as np
import torch

DEFAULT_ATTACK_ITERATIONS = 20
DEFAULT_EXPERT_CONFIG = {
    'experts': 50,
    'min': 0,
    'max': 15,
    'trajectories': [50, 100, 150, 200]
}
DEFAULT_SGD_LABELS_KWARGS = {
    'lr': 150,
    'momentum': 0.5
}
DEFAULT_ATTACK_CONFIG = {
    'iterations': 15,
    'one_hot_temp': 5.,
    'delta': 1.,
    'expert_kwargs': DEFAULT_SGD_KWARGS,
    'label_kwargs': DEFAULT_SGD_LABELS_KWARGS,
    'd_loss': False,
    'd_alpha': 0.5,
    'd_temp': 1.0,
}


def extract_experts(
    expert_config,
    expert_path,
    iterations=None,
    expert_opt_path=None
):
    '''Extracts a list of expert checkpoints for the attack'''
    config = {**DEFAULT_EXPERT_CONFIG, **expert_config}
    expert_starts = []
    expert_opt_starts = []

    for _ in range(iterations or DEFAULT_ATTACK_ITERATIONS):
        for s in config['trajectories']:
            expert = np.random.randint(config['experts'])
            trajectory = np.random.randint(config['min'], config['max']) + 1
            expert_starts.append(expert_path.format(expert, trajectory, str(s)))
            if expert_path:
                expert_opt_starts.append(expert_opt_path.format(expert, trajectory, str(s)))
    return expert_starts, expert_opt_starts


def sgd_step(params, grad, opt_state, opt_params):
    '''Performs a standard step of SGD that is differentiable in the labels'''
    weight_decay = opt_params['weight_decay']
    momentum = opt_params['momentum']
    dampening = opt_params['dampening']
    nesterov = opt_params['nesterov']

    d_p = grad
    if weight_decay != 0:
        d_p = d_p.add(params, alpha=weight_decay)
    if momentum != 0:
        if 'momentum_buffer' not in opt_state:
            buf = opt_state['momentum_buffer'] = torch.zeros_like(params.data)
            buf = buf.mul(momentum).add(d_p)
        else:
            buf = opt_state['momentum_buffer']
            buf = buf.mul(momentum).add(d_p, alpha=1 - dampening)
        if nesterov:
            d_p = d_p.add(buf, alpha=momentum)
        else:
            d_p = buf

    return params.add(d_p, alpha=-opt_params['lr'])


def extract_labels(dataset, label_temp, n_classes=10):
    '''Extracts the labels from a dataset'''
    labels = []
    for _, y in dataset:
        base = np.zeros(n_classes)
        base[y] = label_temp
        labels.append(torch.FloatTensor(base))
    return labels


def coalesce_attack_config(attack_config):
    '''Coalesces the attack config with the default config'''
    expert_kwargs = attack_config.get('expert_kwargs', {})
    labels_kwargs = attack_config.get('labels_kwargs', {})
    attack_config['expert_kwargs'] = {**DEFAULT_SGD_KWARGS, **expert_kwargs}
    attack_config['labels_kwargs'] = {**DEFAULT_SGD_LABELS_KWARGS,
                                      **labels_kwargs}
    return {**DEFAULT_ATTACK_CONFIG, **attack_config}


### `modules/select_flips/run_module.py`


In [ ]:
%%writefile modules/select_flips/run_module.py
"""
Chooses the optimal set of label flips for a given budget.
"""

from pathlib import Path
import sys, glob

import numpy as np

from modules.base_utils.util import extract_toml, slurmify_path


def run(experiment_name, module_name, **kwargs):
    """
    Runs label flip selection and saves a coalesced result.

    :param experiment_name: Name of the experiment in configuration.
    :param module_name: Name of the module in configuration.
    :param kwargs: Additional arguments (such as slurm id).
    """

    slurm_id = kwargs.get('slurm_id', None)

    args = extract_toml(experiment_name, module_name)
    budgets = args.get("budgets", [150, 300, 500, 1000, 1500])
    input_label_glob = slurmify_path(args["input_label_glob"], slurm_id)
    true_labels = slurmify_path(args["true_labels"], slurm_id)
    output_dir = slurmify_path(args["output_dir"], slurm_id)

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Calculate Margins
    print("Calculating margins...")
    distances = []
    all_labels = []
    true = np.load(true_labels)

    for f in glob.glob(input_label_glob):
        labels = np.load(f)

        dists = np.zeros(len(labels))
        inds = labels.argmax(axis=1) != true.argmax(axis=1)
        dists[inds] = labels[inds].max(axis=1) -\
            labels[inds][np.arange(inds.sum()), true[inds].argmax(axis=1)]

        sorted = np.sort(labels[~inds])
        dists[~inds] = sorted[:, -2] - sorted[:, -1]
        distances.append(dists)
        all_labels.append(labels)
    distances = np.stack(distances)
    all_labels = np.stack(all_labels).mean(axis=0)

    # Select flips and save results
    print("Selecting flips...")
    np.save(f'{output_dir}/true.npy', true)
    for n in budgets:
        to_save = true.copy()
        if n != 0:
            idx = np.argsort(distances.min(axis=0))[-n:]
            all_labels[idx] = all_labels[idx] - 50000 * true[idx]
            to_save[idx] = all_labels[idx]
        np.save(f'{output_dir}/{n}.npy', to_save)

if __name__ == "__main__":
    experiment_name, module_name = sys.argv[1], sys.argv[2]
    run(experiment_name, module_name)


### `modules/train_expert/run_module.py`


In [ ]:
%%writefile modules/train_expert/run_module.py
"""
Trains an expert model on a traditionally backdoored dataset.
"""

from pathlib import Path
import sys

from modules.train_expert.utils import checkpoint_callback
from modules.base_utils.datasets import get_matching_datasets, get_n_classes, pick_poisoner
from modules.base_utils.util import extract_toml, load_model, clf_eval, mini_train,\
                                    get_train_info, needs_big_ims, slurmify_path


def run(experiment_name, module_name, **kwargs):
    """
    Runs expert training and saves trajectory.

    :param experiment_name: Name of the experiment in configuration.
    :param module_name: Name of the module in configuration.
    :param kwargs: Additional arguments (such as slurm id).
    """

    slurm_id = kwargs.get('slurm_id', None)
    args = extract_toml(experiment_name, module_name)

    model_flag = args["model"]
    dataset_flag = args["dataset"]
    train_flag = args["trainer"]
    poisoner_flag = args["poisoner"]
    clean_label = args["source_label"]
    target_label = args["target_label"]
    ckpt_iters = args.get("checkpoint_iters")
    train_pct = args.get("train_pct", 1.0)
    batch_size = args.get("batch_size", None)
    epochs = args.get("epochs", None)
    optim_kwargs = args.get("optim_kwargs", {})
    scheduler_kwargs = args.get("scheduler_kwargs", {})
    output_dir = slurmify_path(args["output_dir"], slurm_id)

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    if slurm_id is None:
        slurm_id = "{}"

    # Build datasets
    print("Building datasets...")
    big_ims = needs_big_ims(model_flag)
    poisoner = pick_poisoner(poisoner_flag,
                             dataset_flag,
                             target_label)
    poison_train, _, test, poison_test, _ =\
        get_matching_datasets(dataset_flag, poisoner, clean_label, train_pct=train_pct, big=big_ims)

    # Train expert model
    print("Training expert model...")
    n_classes = get_n_classes(dataset_flag)
    model = load_model(model_flag, n_classes)
    batch_size, epochs, opt, lr_scheduler = get_train_info(
        model.parameters(),
        train_flag,
        batch_size=batch_size,
        epochs=epochs,
        optim_kwargs=optim_kwargs,
        scheduler_kwargs=scheduler_kwargs
    )

    mini_train(
        model=model,
        train_data=poison_train,
        test_data=[test, poison_test.poison_dataset],
        batch_size=batch_size,
        opt=opt,
        scheduler=lr_scheduler,
        epochs=epochs,
        callback=lambda m, o, e, i: checkpoint_callback(m, o, e, i, ckpt_iters, output_dir)
    )

    # Evaluate
    print("Evaluating...")
    clean_test_acc = clf_eval(model, test)[0]
    poison_test_acc = clf_eval(model, poison_test.poison_dataset)[0]
    print(f"{clean_test_acc=}")
    print(f"{poison_test_acc=}")

if __name__ == "__main__":
    experiment_name, module_name = sys.argv[1], sys.argv[2]
    run(experiment_name, module_name)


### `modules/train_expert/utils.py`


In [ ]:
%%writefile modules/train_expert/utils.py
import torch

from modules.base_utils.util import generate_full_path


def checkpoint_callback(model, opt, epoch, iteration, save_iter, output_dir):
    '''Saves model and optimizer state dicts at fixed intervals.'''
    if iteration % save_iter == 0 and iteration != 0:
        checkpoint_path = f'{output_dir}model_{str(epoch)}_{str(iteration)}.pth'
        opt_path = f'{output_dir}model_{str(epoch)}_{str(iteration)}_opt.pth'
        torch.save(model.state_dict(), generate_full_path(checkpoint_path))
        torch.save(opt.state_dict(), generate_full_path(opt_path))

### `modules/train_user/run_module.py`


In [ ]:
%%writefile modules/train_user/run_module.py
"""
Trains a downstream (user) model on a dataset with input labels.
"""

from pathlib import Path
import sys

import torch
import numpy as np

from modules.base_utils.datasets import get_matching_datasets, get_n_classes, pick_poisoner,\
                                        construct_user_dataset
from modules.base_utils.util import extract_toml, get_train_info, mini_train, load_model,\
                                    needs_big_ims, slurmify_path, softmax


def run(experiment_name, module_name, **kwargs):
    """
    Runs user model training and saves metrics.

    :param experiment_name: Name of the experiment in configuration.
    :param module_name: Name of the module in configuration.
    :param kwargs: Additional arguments (such as slurm id).
    """

    slurm_id = kwargs.get('slurm_id', None)
    args = extract_toml(experiment_name, module_name)

    user_model_flag = args["user_model"]
    trainer_flag = args["trainer"]
    dataset_flag = args["dataset"]
    poisoner_flag = args["poisoner"]
    clean_label = args["source_label"]
    target_label = args["target_label"]
    soft = args.get("soft", True)
    batch_size = args.get("batch_size", None)
    epochs = args.get("epochs", None)
    optim_kwargs = args.get("optim_kwargs", {})
    scheduler_kwargs = args.get("scheduler_kwargs", {})
    alpha = args.get("alpha", None)

    input_path = slurmify_path(args["input_labels"], slurm_id)
    true_path = slurmify_path(args.get("true_labels", None), slurm_id)
    output_path = slurmify_path(args["output_dir"], slurm_id)

    Path(output_path).mkdir(parents=True, exist_ok=True)

    # Build datasets
    print("Building datasets...")
    poisoner = pick_poisoner(poisoner_flag, dataset_flag, target_label)

    big_ims = needs_big_ims(user_model_flag)
    _, distillation, test, poison_test, _ =\
        get_matching_datasets(dataset_flag, poisoner, clean_label, big=big_ims)

    labels_syn = torch.tensor(np.load(input_path))        

    if alpha > 0:
        assert true_path is not None
        y_true = torch.tensor(np.load(true_path))
        labels_d = softmax(alpha * y_true + (1 - alpha) * labels_syn)
    else:
        labels_d = softmax(labels_syn)

    if not soft:
        labels_d = labels_d.argmax(dim=1)

    user_dataset = construct_user_dataset(distillation, labels_d)

    # Train user model
    print("Training user model...")
    n_classes = get_n_classes(dataset_flag)
    model_retrain = load_model(user_model_flag, n_classes)
        
    batch_size, epochs, optimizer_retrain, scheduler = get_train_info(
        model_retrain.parameters(), trainer_flag, batch_size,
        epochs, optim_kwargs, scheduler_kwargs
    )

    model_retrain, clean_metrics, poison_metrics = mini_train(
        model=model_retrain,
        train_data=user_dataset,
        test_data=[test, poison_test.poison_dataset],
        batch_size=batch_size,
        opt=optimizer_retrain,
        scheduler=scheduler,
        epochs=epochs,
        record=True
    )

    # Save results
    print("Saving results...")
    np.save(output_path + "paccs.npy", poison_metrics)
    np.save(output_path + "caccs.npy", clean_metrics)
    np.save(output_path + "labels.npy", labels_d.numpy())
    torch.save(model_retrain.state_dict(), output_path + "model.pth")

if __name__ == "__main__":
    experiment_name, module_name = sys.argv[1], sys.argv[2]
    run(experiment_name, module_name)


### `modules/pytorch_cifar/main.py`


In [ ]:
%%writefile modules/pytorch_cifar/main.py
'''Train CIFAR10 with PyTorch.'''
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

import os
import argparse

from models import *
from utils import progress_bar


parser = argparse.ArgumentParser(description='PyTorch CIFAR10 Training')
parser.add_argument('--lr', default=0.1, type=float, help='learning rate')
parser.add_argument('--resume', '-r', action='store_true',
                    help='resume from checkpoint')
args = parser.parse_args()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
best_acc = 0  # best test accuracy
start_epoch = 0  # start from epoch 0 or last checkpoint epoch

# Data
print('==> Preparing data..')
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=100, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

# Model
print('==> Building model..')
# net = VGG('VGG19')
# net = ResNet18()
# net = PreActResNet18()
# net = GoogLeNet()
# net = DenseNet121()
# net = ResNeXt29_2x64d()
# net = MobileNet()
# net = MobileNetV2()
# net = DPN92()
# net = ShuffleNetG2()
# net = SENet18()
# net = ShuffleNetV2(1)
# net = EfficientNetB0()
# net = RegNetX_200MF()
net = SimpleDLA()
net = net.to(device)
if device == 'cuda':
    net = torch.nn.DataParallel(net)
    cudnn.benchmark = True

if args.resume:
    # Load checkpoint.
    print('==> Resuming from checkpoint..')
    assert os.path.isdir('checkpoint'), 'Error: no checkpoint directory found!'
    checkpoint = torch.load('./checkpoint/ckpt.pth')
    net.load_state_dict(checkpoint['net'])
    best_acc = checkpoint['acc']
    start_epoch = checkpoint['epoch']

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=args.lr,
                      momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)


# Training
def train(epoch):
    print('\nEpoch: %d' % epoch)
    net.train()
    train_loss = 0
    correct = 0
    total = 0
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        progress_bar(batch_idx, len(trainloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d)'
                     % (train_loss/(batch_idx+1), 100.*correct/total, correct, total))


def test(epoch):
    global best_acc
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            progress_bar(batch_idx, len(testloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d)'
                         % (test_loss/(batch_idx+1), 100.*correct/total, correct, total))

    # Save checkpoint.
    acc = 100.*correct/total
    if acc > best_acc:
        print('Saving..')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
        }
        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        torch.save(state, './checkpoint/ckpt.pth')
        best_acc = acc


for epoch in range(start_epoch, start_epoch+200):
    train(epoch)
    test(epoch)
    scheduler.step()


### `modules/pytorch_cifar/utils.py`


In [ ]:
%%writefile modules/pytorch_cifar/utils.py
'''Some helper functions for PyTorch, including:
    - get_mean_and_std: calculate the mean and std value of dataset.
    - msr_init: net parameter initialization.
    - progress_bar: progress bar mimic xlua.progress.
'''
import os
import sys
import time
import math

import torch.nn as nn
import torch.nn.init as init


def get_mean_and_std(dataset):
    '''Compute the mean and std value of dataset.'''
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True, num_workers=2)
    mean = torch.zeros(3)
    std = torch.zeros(3)
    print('==> Computing mean and std..')
    for inputs, targets in dataloader:
        for i in range(3):
            mean[i] += inputs[:,i,:,:].mean()
            std[i] += inputs[:,i,:,:].std()
    mean.div_(len(dataset))
    std.div_(len(dataset))
    return mean, std

def init_params(net):
    '''Init layer parameters.'''
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            init.kaiming_normal(m.weight, mode='fan_out')
            if m.bias:
                init.constant(m.bias, 0)
        elif isinstance(m, nn.BatchNorm2d):
            init.constant(m.weight, 1)
            init.constant(m.bias, 0)
        elif isinstance(m, nn.Linear):
            init.normal(m.weight, std=1e-3)
            if m.bias:
                init.constant(m.bias, 0)


_, term_width = os.popen('stty size', 'r').read().split()
term_width = int(term_width)

TOTAL_BAR_LENGTH = 65.
last_time = time.time()
begin_time = last_time
def progress_bar(current, total, msg=None):
    global last_time, begin_time
    if current == 0:
        begin_time = time.time()  # Reset for new bar.

    cur_len = int(TOTAL_BAR_LENGTH*current/total)
    rest_len = int(TOTAL_BAR_LENGTH - cur_len) - 1

    sys.stdout.write(' [')
    for i in range(cur_len):
        sys.stdout.write('=')
    sys.stdout.write('>')
    for i in range(rest_len):
        sys.stdout.write('.')
    sys.stdout.write(']')

    cur_time = time.time()
    step_time = cur_time - last_time
    last_time = cur_time
    tot_time = cur_time - begin_time

    L = []
    L.append('  Step: %s' % format_time(step_time))
    L.append(' | Tot: %s' % format_time(tot_time))
    if msg:
        L.append(' | ' + msg)

    msg = ''.join(L)
    sys.stdout.write(msg)
    for i in range(term_width-int(TOTAL_BAR_LENGTH)-len(msg)-3):
        sys.stdout.write(' ')

    # Go back to the center of the bar.
    for i in range(term_width-int(TOTAL_BAR_LENGTH/2)+2):
        sys.stdout.write('\b')
    sys.stdout.write(' %d/%d ' % (current+1, total))

    if current < total-1:
        sys.stdout.write('\r')
    else:
        sys.stdout.write('\n')
    sys.stdout.flush()

def format_time(seconds):
    days = int(seconds / 3600/24)
    seconds = seconds - days*3600*24
    hours = int(seconds / 3600)
    seconds = seconds - hours*3600
    minutes = int(seconds / 60)
    seconds = seconds - minutes*60
    secondsf = int(seconds)
    seconds = seconds - secondsf
    millis = int(seconds*1000)

    f = ''
    i = 1
    if days > 0:
        f += str(days) + 'D'
        i += 1
    if hours > 0 and i <= 2:
        f += str(hours) + 'h'
        i += 1
    if minutes > 0 and i <= 2:
        f += str(minutes) + 'm'
        i += 1
    if secondsf > 0 and i <= 2:
        f += str(secondsf) + 's'
        i += 1
    if millis > 0 and i <= 2:
        f += str(millis) + 'ms'
        i += 1
    if f == '':
        f = '0ms'
    return f


### `modules/pytorch_cifar/README.md`


In [ ]:
%%writefile modules/pytorch_cifar/README.md
# Train CIFAR10 with PyTorch

I'm playing with [PyTorch](http://pytorch.org/) on the CIFAR10 dataset.

## Prerequisites
- Python 3.6+
- PyTorch 1.0+

## Training
```
# Start training with: 
python main.py

# You can manually resume the training with: 
python main.py --resume --lr=0.01
```

## Accuracy
| Model             | Acc.        |
| ----------------- | ----------- |
| [VGG16](https://arxiv.org/abs/1409.1556)              | 92.64%      |
| [ResNet18](https://arxiv.org/abs/1512.03385)          | 93.02%      |
| [ResNet50](https://arxiv.org/abs/1512.03385)          | 93.62%      |
| [ResNet101](https://arxiv.org/abs/1512.03385)         | 93.75%      |
| [RegNetX_200MF](https://arxiv.org/abs/2003.13678)     | 94.24%      |
| [RegNetY_400MF](https://arxiv.org/abs/2003.13678)     | 94.29%      |
| [MobileNetV2](https://arxiv.org/abs/1801.04381)       | 94.43%      |
| [ResNeXt29(32x4d)](https://arxiv.org/abs/1611.05431)  | 94.73%      |
| [ResNeXt29(2x64d)](https://arxiv.org/abs/1611.05431)  | 94.82%      |
| [SimpleDLA](https://arxiv.org/abs/1707.064)           | 94.89%      |
| [DenseNet121](https://arxiv.org/abs/1608.06993)       | 95.04%      |
| [PreActResNet18](https://arxiv.org/abs/1603.05027)    | 95.11%      |
| [DPN92](https://arxiv.org/abs/1707.01629)             | 95.16%      |
| [DLA](https://arxiv.org/pdf/1707.06484.pdf)           | 95.47%      |



### `modules/pytorch_cifar/LICENSE`


In [ ]:
%%writefile modules/pytorch_cifar/LICENSE
MIT License

Copyright (c) 2017 liukuang

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


### `modules/pytorch_cifar/models/__init__.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/__init__.py
from .vgg import *
from .dpn import *
from .lenet import *
from .senet import *
from .pnasnet import *
from .densenet import *
from .googlenet import *
from .shufflenet import *
from .shufflenetv2 import *
from .resnet import *
from .resnext import *
from .preact_resnet import *
from .mobilenet import *
from .mobilenetv2 import *
from .efficientnet import *
from .regnet import *
from .dla_simple import *
from .dla import *


### `modules/pytorch_cifar/models/densenet.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/densenet.py
'''DenseNet in PyTorch.'''
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class Bottleneck(nn.Module):
    def __init__(self, in_planes, growth_rate):
        super(Bottleneck, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, 4*growth_rate, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(4*growth_rate)
        self.conv2 = nn.Conv2d(4*growth_rate, growth_rate, kernel_size=3, padding=1, bias=False)

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x)))
        out = self.conv2(F.relu(self.bn2(out)))
        out = torch.cat([out,x], 1)
        return out


class Transition(nn.Module):
    def __init__(self, in_planes, out_planes):
        super(Transition, self).__init__()
        self.bn = nn.BatchNorm2d(in_planes)
        self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=1, bias=False)

    def forward(self, x):
        out = self.conv(F.relu(self.bn(x)))
        out = F.avg_pool2d(out, 2)
        return out


class DenseNet(nn.Module):
    def __init__(self, block, nblocks, growth_rate=12, reduction=0.5, num_classes=10):
        super(DenseNet, self).__init__()
        self.growth_rate = growth_rate

        num_planes = 2*growth_rate
        self.conv1 = nn.Conv2d(3, num_planes, kernel_size=3, padding=1, bias=False)

        self.dense1 = self._make_dense_layers(block, num_planes, nblocks[0])
        num_planes += nblocks[0]*growth_rate
        out_planes = int(math.floor(num_planes*reduction))
        self.trans1 = Transition(num_planes, out_planes)
        num_planes = out_planes

        self.dense2 = self._make_dense_layers(block, num_planes, nblocks[1])
        num_planes += nblocks[1]*growth_rate
        out_planes = int(math.floor(num_planes*reduction))
        self.trans2 = Transition(num_planes, out_planes)
        num_planes = out_planes

        self.dense3 = self._make_dense_layers(block, num_planes, nblocks[2])
        num_planes += nblocks[2]*growth_rate
        out_planes = int(math.floor(num_planes*reduction))
        self.trans3 = Transition(num_planes, out_planes)
        num_planes = out_planes

        self.dense4 = self._make_dense_layers(block, num_planes, nblocks[3])
        num_planes += nblocks[3]*growth_rate

        self.bn = nn.BatchNorm2d(num_planes)
        self.linear = nn.Linear(num_planes, num_classes)

    def _make_dense_layers(self, block, in_planes, nblock):
        layers = []
        for i in range(nblock):
            layers.append(block(in_planes, self.growth_rate))
            in_planes += self.growth_rate
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.trans1(self.dense1(out))
        out = self.trans2(self.dense2(out))
        out = self.trans3(self.dense3(out))
        out = self.dense4(out)
        out = F.avg_pool2d(F.relu(self.bn(out)), 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def DenseNet121():
    return DenseNet(Bottleneck, [6,12,24,16], growth_rate=32)

def DenseNet169():
    return DenseNet(Bottleneck, [6,12,32,32], growth_rate=32)

def DenseNet201():
    return DenseNet(Bottleneck, [6,12,48,32], growth_rate=32)

def DenseNet161():
    return DenseNet(Bottleneck, [6,12,36,24], growth_rate=48)

def densenet_cifar():
    return DenseNet(Bottleneck, [6,12,24,16], growth_rate=12)

def test():
    net = densenet_cifar()
    x = torch.randn(1,3,32,32)
    y = net(x)
    print(y)

# test()


### `modules/pytorch_cifar/models/dla.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/dla.py
'''DLA in PyTorch.

Reference:
    Deep Layer Aggregation. https://arxiv.org/abs/1707.06484
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class Root(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=1):
        super(Root, self).__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size,
            stride=1, padding=(kernel_size - 1) // 2, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, xs):
        x = torch.cat(xs, 1)
        out = F.relu(self.bn(self.conv(x)))
        return out


class Tree(nn.Module):
    def __init__(self, block, in_channels, out_channels, level=1, stride=1):
        super(Tree, self).__init__()
        self.level = level
        if level == 1:
            self.root = Root(2*out_channels, out_channels)
            self.left_node = block(in_channels, out_channels, stride=stride)
            self.right_node = block(out_channels, out_channels, stride=1)
        else:
            self.root = Root((level+2)*out_channels, out_channels)
            for i in reversed(range(1, level)):
                subtree = Tree(block, in_channels, out_channels,
                               level=i, stride=stride)
                self.__setattr__('level_%d' % i, subtree)
            self.prev_root = block(in_channels, out_channels, stride=stride)
            self.left_node = block(out_channels, out_channels, stride=1)
            self.right_node = block(out_channels, out_channels, stride=1)

    def forward(self, x):
        xs = [self.prev_root(x)] if self.level > 1 else []
        for i in reversed(range(1, self.level)):
            level_i = self.__getattr__('level_%d' % i)
            x = level_i(x)
            xs.append(x)
        x = self.left_node(x)
        xs.append(x)
        x = self.right_node(x)
        xs.append(x)
        out = self.root(xs)
        return out


class DLA(nn.Module):
    def __init__(self, block=BasicBlock, num_classes=10):
        super(DLA, self).__init__()
        self.base = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(True)
        )

        self.layer1 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(True)
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(True)
        )

        self.layer3 = Tree(block,  32,  64, level=1, stride=1)
        self.layer4 = Tree(block,  64, 128, level=2, stride=2)
        self.layer5 = Tree(block, 128, 256, level=2, stride=2)
        self.layer6 = Tree(block, 256, 512, level=1, stride=2)
        self.linear = nn.Linear(512, num_classes)

    def forward(self, x):
        out = self.base(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.layer5(out)
        out = self.layer6(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def test():
    net = DLA()
    print(net)
    x = torch.randn(1, 3, 32, 32)
    y = net(x)
    print(y.size())


if __name__ == '__main__':
    test()


### `modules/pytorch_cifar/models/dla_simple.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/dla_simple.py
'''Simplified version of DLA in PyTorch.

Note this implementation is not identical to the original paper version.
But it seems works fine.

See dla.py for the original paper version.

Reference:
    Deep Layer Aggregation. https://arxiv.org/abs/1707.06484
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class Root(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=1):
        super(Root, self).__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size,
            stride=1, padding=(kernel_size - 1) // 2, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, xs):
        x = torch.cat(xs, 1)
        out = F.relu(self.bn(self.conv(x)))
        return out


class Tree(nn.Module):
    def __init__(self, block, in_channels, out_channels, level=1, stride=1):
        super(Tree, self).__init__()
        self.root = Root(2*out_channels, out_channels)
        if level == 1:
            self.left_tree = block(in_channels, out_channels, stride=stride)
            self.right_tree = block(out_channels, out_channels, stride=1)
        else:
            self.left_tree = Tree(block, in_channels,
                                  out_channels, level=level-1, stride=stride)
            self.right_tree = Tree(block, out_channels,
                                   out_channels, level=level-1, stride=1)

    def forward(self, x):
        out1 = self.left_tree(x)
        out2 = self.right_tree(out1)
        out = self.root([out1, out2])
        return out


class SimpleDLA(nn.Module):
    def __init__(self, block=BasicBlock, num_classes=10):
        super(SimpleDLA, self).__init__()
        self.base = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(True)
        )

        self.layer1 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(True)
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(True)
        )

        self.layer3 = Tree(block,  32,  64, level=1, stride=1)
        self.layer4 = Tree(block,  64, 128, level=2, stride=2)
        self.layer5 = Tree(block, 128, 256, level=2, stride=2)
        self.layer6 = Tree(block, 256, 512, level=1, stride=2)
        self.linear = nn.Linear(512, num_classes)

    def forward(self, x):
        out = self.base(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.layer5(out)
        out = self.layer6(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def test():
    net = SimpleDLA()
    print(net)
    x = torch.randn(1, 3, 32, 32)
    y = net(x)
    print(y.size())


if __name__ == '__main__':
    test()


### `modules/pytorch_cifar/models/dpn.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/dpn.py
'''Dual Path Networks in PyTorch.'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class Bottleneck(nn.Module):
    def __init__(self, last_planes, in_planes, out_planes, dense_depth, stride, first_layer):
        super(Bottleneck, self).__init__()
        self.out_planes = out_planes
        self.dense_depth = dense_depth

        self.conv1 = nn.Conv2d(last_planes, in_planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv2 = nn.Conv2d(in_planes, in_planes, kernel_size=3, stride=stride, padding=1, groups=32, bias=False)
        self.bn2 = nn.BatchNorm2d(in_planes)
        self.conv3 = nn.Conv2d(in_planes, out_planes+dense_depth, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_planes+dense_depth)

        self.shortcut = nn.Sequential()
        if first_layer:
            self.shortcut = nn.Sequential(
                nn.Conv2d(last_planes, out_planes+dense_depth, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_planes+dense_depth)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        x = self.shortcut(x)
        d = self.out_planes
        out = torch.cat([x[:,:d,:,:]+out[:,:d,:,:], x[:,d:,:,:], out[:,d:,:,:]], 1)
        out = F.relu(out)
        return out


class DPN(nn.Module):
    def __init__(self, cfg):
        super(DPN, self).__init__()
        in_planes, out_planes = cfg['in_planes'], cfg['out_planes']
        num_blocks, dense_depth = cfg['num_blocks'], cfg['dense_depth']

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.last_planes = 64
        self.layer1 = self._make_layer(in_planes[0], out_planes[0], num_blocks[0], dense_depth[0], stride=1)
        self.layer2 = self._make_layer(in_planes[1], out_planes[1], num_blocks[1], dense_depth[1], stride=2)
        self.layer3 = self._make_layer(in_planes[2], out_planes[2], num_blocks[2], dense_depth[2], stride=2)
        self.layer4 = self._make_layer(in_planes[3], out_planes[3], num_blocks[3], dense_depth[3], stride=2)
        self.linear = nn.Linear(out_planes[3]+(num_blocks[3]+1)*dense_depth[3], 10)

    def _make_layer(self, in_planes, out_planes, num_blocks, dense_depth, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for i,stride in enumerate(strides):
            layers.append(Bottleneck(self.last_planes, in_planes, out_planes, dense_depth, stride, i==0))
            self.last_planes = out_planes + (i+2) * dense_depth
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def DPN26():
    cfg = {
        'in_planes': (96,192,384,768),
        'out_planes': (256,512,1024,2048),
        'num_blocks': (2,2,2,2),
        'dense_depth': (16,32,24,128)
    }
    return DPN(cfg)

def DPN92():
    cfg = {
        'in_planes': (96,192,384,768),
        'out_planes': (256,512,1024,2048),
        'num_blocks': (3,4,20,3),
        'dense_depth': (16,32,24,128)
    }
    return DPN(cfg)


def test():
    net = DPN92()
    x = torch.randn(1,3,32,32)
    y = net(x)
    print(y)

# test()


### `modules/pytorch_cifar/models/efficientnet.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/efficientnet.py
'''EfficientNet in PyTorch.

Paper: "EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks".

Reference: https://github.com/keras-team/keras-applications/blob/master/keras_applications/efficientnet.py
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


def swish(x):
    return x * x.sigmoid()


def drop_connect(x, drop_ratio):
    keep_ratio = 1.0 - drop_ratio
    mask = torch.empty([x.shape[0], 1, 1, 1], dtype=x.dtype, device=x.device)
    mask.bernoulli_(keep_ratio)
    x.div_(keep_ratio)
    x.mul_(mask)
    return x


class SE(nn.Module):
    '''Squeeze-and-Excitation block with Swish.'''

    def __init__(self, in_channels, se_channels):
        super(SE, self).__init__()
        self.se1 = nn.Conv2d(in_channels, se_channels,
                             kernel_size=1, bias=True)
        self.se2 = nn.Conv2d(se_channels, in_channels,
                             kernel_size=1, bias=True)

    def forward(self, x):
        out = F.adaptive_avg_pool2d(x, (1, 1))
        out = swish(self.se1(out))
        out = self.se2(out).sigmoid()
        out = x * out
        return out


class Block(nn.Module):
    '''expansion + depthwise + pointwise + squeeze-excitation'''

    def __init__(self,
                 in_channels,
                 out_channels,
                 kernel_size,
                 stride,
                 expand_ratio=1,
                 se_ratio=0.,
                 drop_rate=0.):
        super(Block, self).__init__()
        self.stride = stride
        self.drop_rate = drop_rate
        self.expand_ratio = expand_ratio

        # Expansion
        channels = expand_ratio * in_channels
        self.conv1 = nn.Conv2d(in_channels,
                               channels,
                               kernel_size=1,
                               stride=1,
                               padding=0,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(channels)

        # Depthwise conv
        self.conv2 = nn.Conv2d(channels,
                               channels,
                               kernel_size=kernel_size,
                               stride=stride,
                               padding=(1 if kernel_size == 3 else 2),
                               groups=channels,
                               bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

        # SE layers
        se_channels = int(in_channels * se_ratio)
        self.se = SE(channels, se_channels)

        # Output
        self.conv3 = nn.Conv2d(channels,
                               out_channels,
                               kernel_size=1,
                               stride=1,
                               padding=0,
                               bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)

        # Skip connection if in and out shapes are the same (MV-V2 style)
        self.has_skip = (stride == 1) and (in_channels == out_channels)

    def forward(self, x):
        out = x if self.expand_ratio == 1 else swish(self.bn1(self.conv1(x)))
        out = swish(self.bn2(self.conv2(out)))
        out = self.se(out)
        out = self.bn3(self.conv3(out))
        if self.has_skip:
            if self.training and self.drop_rate > 0:
                out = drop_connect(out, self.drop_rate)
            out = out + x
        return out


class EfficientNet(nn.Module):
    def __init__(self, cfg, num_classes=10):
        super(EfficientNet, self).__init__()
        self.cfg = cfg
        self.conv1 = nn.Conv2d(3,
                               32,
                               kernel_size=3,
                               stride=1,
                               padding=1,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.layers = self._make_layers(in_channels=32)
        self.linear = nn.Linear(cfg['out_channels'][-1], num_classes)

    def _make_layers(self, in_channels):
        layers = []
        cfg = [self.cfg[k] for k in ['expansion', 'out_channels', 'num_blocks', 'kernel_size',
                                     'stride']]
        b = 0
        blocks = sum(self.cfg['num_blocks'])
        for expansion, out_channels, num_blocks, kernel_size, stride in zip(*cfg):
            strides = [stride] + [1] * (num_blocks - 1)
            for stride in strides:
                drop_rate = self.cfg['drop_connect_rate'] * b / blocks
                layers.append(
                    Block(in_channels,
                          out_channels,
                          kernel_size,
                          stride,
                          expansion,
                          se_ratio=0.25,
                          drop_rate=drop_rate))
                in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = swish(self.bn1(self.conv1(x)))
        out = self.layers(out)
        out = F.adaptive_avg_pool2d(out, 1)
        out = out.view(out.size(0), -1)
        dropout_rate = self.cfg['dropout_rate']
        if self.training and dropout_rate > 0:
            out = F.dropout(out, p=dropout_rate)
        out = self.linear(out)
        return out


def EfficientNetB0():
    cfg = {
        'num_blocks': [1, 2, 2, 3, 3, 4, 1],
        'expansion': [1, 6, 6, 6, 6, 6, 6],
        'out_channels': [16, 24, 40, 80, 112, 192, 320],
        'kernel_size': [3, 3, 5, 3, 5, 5, 3],
        'stride': [1, 2, 2, 2, 1, 2, 1],
        'dropout_rate': 0.2,
        'drop_connect_rate': 0.2,
    }
    return EfficientNet(cfg)


def test():
    net = EfficientNetB0()
    x = torch.randn(2, 3, 32, 32)
    y = net(x)
    print(y.shape)


if __name__ == '__main__':
    test()


### `modules/pytorch_cifar/models/googlenet.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/googlenet.py
'''GoogLeNet with PyTorch.'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class Inception(nn.Module):
    def __init__(self, in_planes, n1x1, n3x3red, n3x3, n5x5red, n5x5, pool_planes):
        super(Inception, self).__init__()
        # 1x1 conv branch
        self.b1 = nn.Sequential(
            nn.Conv2d(in_planes, n1x1, kernel_size=1),
            nn.BatchNorm2d(n1x1),
            nn.ReLU(True),
        )

        # 1x1 conv -> 3x3 conv branch
        self.b2 = nn.Sequential(
            nn.Conv2d(in_planes, n3x3red, kernel_size=1),
            nn.BatchNorm2d(n3x3red),
            nn.ReLU(True),
            nn.Conv2d(n3x3red, n3x3, kernel_size=3, padding=1),
            nn.BatchNorm2d(n3x3),
            nn.ReLU(True),
        )

        # 1x1 conv -> 5x5 conv branch
        self.b3 = nn.Sequential(
            nn.Conv2d(in_planes, n5x5red, kernel_size=1),
            nn.BatchNorm2d(n5x5red),
            nn.ReLU(True),
            nn.Conv2d(n5x5red, n5x5, kernel_size=3, padding=1),
            nn.BatchNorm2d(n5x5),
            nn.ReLU(True),
            nn.Conv2d(n5x5, n5x5, kernel_size=3, padding=1),
            nn.BatchNorm2d(n5x5),
            nn.ReLU(True),
        )

        # 3x3 pool -> 1x1 conv branch
        self.b4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_planes, pool_planes, kernel_size=1),
            nn.BatchNorm2d(pool_planes),
            nn.ReLU(True),
        )

    def forward(self, x):
        y1 = self.b1(x)
        y2 = self.b2(x)
        y3 = self.b3(x)
        y4 = self.b4(x)
        return torch.cat([y1,y2,y3,y4], 1)


class GoogLeNet(nn.Module):
    def __init__(self):
        super(GoogLeNet, self).__init__()
        self.pre_layers = nn.Sequential(
            nn.Conv2d(3, 192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(True),
        )

        self.a3 = Inception(192,  64,  96, 128, 16, 32, 32)
        self.b3 = Inception(256, 128, 128, 192, 32, 96, 64)

        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)

        self.a4 = Inception(480, 192,  96, 208, 16,  48,  64)
        self.b4 = Inception(512, 160, 112, 224, 24,  64,  64)
        self.c4 = Inception(512, 128, 128, 256, 24,  64,  64)
        self.d4 = Inception(512, 112, 144, 288, 32,  64,  64)
        self.e4 = Inception(528, 256, 160, 320, 32, 128, 128)

        self.a5 = Inception(832, 256, 160, 320, 32, 128, 128)
        self.b5 = Inception(832, 384, 192, 384, 48, 128, 128)

        self.avgpool = nn.AvgPool2d(8, stride=1)
        self.linear = nn.Linear(1024, 10)

    def forward(self, x):
        out = self.pre_layers(x)
        out = self.a3(out)
        out = self.b3(out)
        out = self.maxpool(out)
        out = self.a4(out)
        out = self.b4(out)
        out = self.c4(out)
        out = self.d4(out)
        out = self.e4(out)
        out = self.maxpool(out)
        out = self.a5(out)
        out = self.b5(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def test():
    net = GoogLeNet()
    x = torch.randn(1,3,32,32)
    y = net(x)
    print(y.size())

# test()


### `modules/pytorch_cifar/models/lenet.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/lenet.py
'''LeNet in PyTorch.'''
import torch.nn as nn
import torch.nn.functional as F

class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1   = nn.Linear(16*5*5, 120)
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)

    def forward(self, x):
        out = F.relu(self.conv1(x))
        out = F.max_pool2d(out, 2)
        out = F.relu(self.conv2(out))
        out = F.max_pool2d(out, 2)
        out = out.view(out.size(0), -1)
        out = F.relu(self.fc1(out))
        out = F.relu(self.fc2(out))
        out = self.fc3(out)
        return out


### `modules/pytorch_cifar/models/mobilenet.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/mobilenet.py
'''MobileNet in PyTorch.

See the paper "MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications"
for more details.
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    '''Depthwise conv + Pointwise conv'''
    def __init__(self, in_planes, out_planes, stride=1):
        super(Block, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, in_planes, kernel_size=3, stride=stride, padding=1, groups=in_planes, bias=False)
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv2 = nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        return out


class MobileNet(nn.Module):
    # (128,2) means conv planes=128, conv stride=2, by default conv stride=1
    cfg = [64, (128,2), 128, (256,2), 256, (512,2), 512, 512, 512, 512, 512, (1024,2), 1024]

    def __init__(self, num_classes=10):
        super(MobileNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.layers = self._make_layers(in_planes=32)
        self.linear = nn.Linear(1024, num_classes)

    def _make_layers(self, in_planes):
        layers = []
        for x in self.cfg:
            out_planes = x if isinstance(x, int) else x[0]
            stride = 1 if isinstance(x, int) else x[1]
            layers.append(Block(in_planes, out_planes, stride))
            in_planes = out_planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layers(out)
        out = F.avg_pool2d(out, 2)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def test():
    net = MobileNet()
    x = torch.randn(1,3,32,32)
    y = net(x)
    print(y.size())

# test()


### `modules/pytorch_cifar/models/mobilenetv2.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/mobilenetv2.py
'''MobileNetV2 in PyTorch.

See the paper "Inverted Residuals and Linear Bottlenecks:
Mobile Networks for Classification, Detection and Segmentation" for more details.
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    '''expand + depthwise + pointwise'''
    def __init__(self, in_planes, out_planes, expansion, stride):
        super(Block, self).__init__()
        self.stride = stride

        planes = expansion * in_planes
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, groups=planes, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, out_planes, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn3 = nn.BatchNorm2d(out_planes)

        self.shortcut = nn.Sequential()
        if stride == 1 and in_planes != out_planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(out_planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = out + self.shortcut(x) if self.stride==1 else out
        return out


class MobileNetV2(nn.Module):
    # (expansion, out_planes, num_blocks, stride)
    cfg = [(1,  16, 1, 1),
           (6,  24, 2, 1),  # NOTE: change stride 2 -> 1 for CIFAR10
           (6,  32, 3, 2),
           (6,  64, 4, 2),
           (6,  96, 3, 1),
           (6, 160, 3, 2),
           (6, 320, 1, 1)]

    def __init__(self, num_classes=10):
        super(MobileNetV2, self).__init__()
        # NOTE: change conv1 stride 2 -> 1 for CIFAR10
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.layers = self._make_layers(in_planes=32)
        self.conv2 = nn.Conv2d(320, 1280, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn2 = nn.BatchNorm2d(1280)
        self.linear = nn.Linear(1280, num_classes)

    def _make_layers(self, in_planes):
        layers = []
        for expansion, out_planes, num_blocks, stride in self.cfg:
            strides = [stride] + [1]*(num_blocks-1)
            for stride in strides:
                layers.append(Block(in_planes, out_planes, expansion, stride))
                in_planes = out_planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layers(out)
        out = F.relu(self.bn2(self.conv2(out)))
        # NOTE: change pooling kernel_size 7 -> 4 for CIFAR10
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def test():
    net = MobileNetV2()
    x = torch.randn(2,3,32,32)
    y = net(x)
    print(y.size())

# test()


### `modules/pytorch_cifar/models/pnasnet.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/pnasnet.py
'''PNASNet in PyTorch.

Paper: Progressive Neural Architecture Search
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class SepConv(nn.Module):
    '''Separable Convolution.'''
    def __init__(self, in_planes, out_planes, kernel_size, stride):
        super(SepConv, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, out_planes,
                               kernel_size, stride,
                               padding=(kernel_size-1)//2,
                               bias=False, groups=in_planes)
        self.bn1 = nn.BatchNorm2d(out_planes)

    def forward(self, x):
        return self.bn1(self.conv1(x))


class CellA(nn.Module):
    def __init__(self, in_planes, out_planes, stride=1):
        super(CellA, self).__init__()
        self.stride = stride
        self.sep_conv1 = SepConv(in_planes, out_planes, kernel_size=7, stride=stride)
        if stride==2:
            self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=1, padding=0, bias=False)
            self.bn1 = nn.BatchNorm2d(out_planes)

    def forward(self, x):
        y1 = self.sep_conv1(x)
        y2 = F.max_pool2d(x, kernel_size=3, stride=self.stride, padding=1)
        if self.stride==2:
            y2 = self.bn1(self.conv1(y2))
        return F.relu(y1+y2)

class CellB(nn.Module):
    def __init__(self, in_planes, out_planes, stride=1):
        super(CellB, self).__init__()
        self.stride = stride
        # Left branch
        self.sep_conv1 = SepConv(in_planes, out_planes, kernel_size=7, stride=stride)
        self.sep_conv2 = SepConv(in_planes, out_planes, kernel_size=3, stride=stride)
        # Right branch
        self.sep_conv3 = SepConv(in_planes, out_planes, kernel_size=5, stride=stride)
        if stride==2:
            self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=1, padding=0, bias=False)
            self.bn1 = nn.BatchNorm2d(out_planes)
        # Reduce channels
        self.conv2 = nn.Conv2d(2*out_planes, out_planes, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)

    def forward(self, x):
        # Left branch
        y1 = self.sep_conv1(x)
        y2 = self.sep_conv2(x)
        # Right branch
        y3 = F.max_pool2d(x, kernel_size=3, stride=self.stride, padding=1)
        if self.stride==2:
            y3 = self.bn1(self.conv1(y3))
        y4 = self.sep_conv3(x)
        # Concat & reduce channels
        b1 = F.relu(y1+y2)
        b2 = F.relu(y3+y4)
        y = torch.cat([b1,b2], 1)
        return F.relu(self.bn2(self.conv2(y)))

class PNASNet(nn.Module):
    def __init__(self, cell_type, num_cells, num_planes):
        super(PNASNet, self).__init__()
        self.in_planes = num_planes
        self.cell_type = cell_type

        self.conv1 = nn.Conv2d(3, num_planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(num_planes)

        self.layer1 = self._make_layer(num_planes, num_cells=6)
        self.layer2 = self._downsample(num_planes*2)
        self.layer3 = self._make_layer(num_planes*2, num_cells=6)
        self.layer4 = self._downsample(num_planes*4)
        self.layer5 = self._make_layer(num_planes*4, num_cells=6)

        self.linear = nn.Linear(num_planes*4, 10)

    def _make_layer(self, planes, num_cells):
        layers = []
        for _ in range(num_cells):
            layers.append(self.cell_type(self.in_planes, planes, stride=1))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def _downsample(self, planes):
        layer = self.cell_type(self.in_planes, planes, stride=2)
        self.in_planes = planes
        return layer

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.layer5(out)
        out = F.avg_pool2d(out, 8)
        out = self.linear(out.view(out.size(0), -1))
        return out


def PNASNetA():
    return PNASNet(CellA, num_cells=6, num_planes=44)

def PNASNetB():
    return PNASNet(CellB, num_cells=6, num_planes=32)


def test():
    net = PNASNetB()
    x = torch.randn(1,3,32,32)
    y = net(x)
    print(y)

# test()


### `modules/pytorch_cifar/models/preact_resnet.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/preact_resnet.py
'''Pre-activation ResNet in PyTorch.

Reference:
[1] Kaiming He, Xiangyu Zhang, Shaoqing Ren, Jian Sun
    Identity Mappings in Deep Residual Networks. arXiv:1603.05027
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class PreActBlock(nn.Module):
    '''Pre-activation version of the BasicBlock.'''
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(PreActBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False)
            )

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        out += shortcut
        return out


class PreActBottleneck(nn.Module):
    '''Pre-activation version of the original Bottleneck module.'''
    expansion = 4

    def __init__(self, in_planes, planes, stride=1):
        super(PreActBottleneck, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, self.expansion*planes, kernel_size=1, bias=False)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False)
            )

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        out = self.conv3(F.relu(self.bn3(out)))
        out += shortcut
        return out


class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(PreActResNet, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512*block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def PreActResNet18():
    return PreActResNet(PreActBlock, [2,2,2,2])

def PreActResNet34():
    return PreActResNet(PreActBlock, [3,4,6,3])

def PreActResNet50():
    return PreActResNet(PreActBottleneck, [3,4,6,3])

def PreActResNet101():
    return PreActResNet(PreActBottleneck, [3,4,23,3])

def PreActResNet152():
    return PreActResNet(PreActBottleneck, [3,8,36,3])


def test():
    net = PreActResNet18()
    y = net((torch.randn(1,3,32,32)))
    print(y.size())

# test()


### `modules/pytorch_cifar/models/regnet.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/regnet.py
'''RegNet in PyTorch.

Paper: "Designing Network Design Spaces".

Reference: https://github.com/keras-team/keras-applications/blob/master/keras_applications/efficientnet.py
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class SE(nn.Module):
    '''Squeeze-and-Excitation block.'''

    def __init__(self, in_planes, se_planes):
        super(SE, self).__init__()
        self.se1 = nn.Conv2d(in_planes, se_planes, kernel_size=1, bias=True)
        self.se2 = nn.Conv2d(se_planes, in_planes, kernel_size=1, bias=True)

    def forward(self, x):
        out = F.adaptive_avg_pool2d(x, (1, 1))
        out = F.relu(self.se1(out))
        out = self.se2(out).sigmoid()
        out = x * out
        return out


class Block(nn.Module):
    def __init__(self, w_in, w_out, stride, group_width, bottleneck_ratio, se_ratio):
        super(Block, self).__init__()
        # 1x1
        w_b = int(round(w_out * bottleneck_ratio))
        self.conv1 = nn.Conv2d(w_in, w_b, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(w_b)
        # 3x3
        num_groups = w_b // group_width
        self.conv2 = nn.Conv2d(w_b, w_b, kernel_size=3,
                               stride=stride, padding=1, groups=num_groups, bias=False)
        self.bn2 = nn.BatchNorm2d(w_b)
        # se
        self.with_se = se_ratio > 0
        if self.with_se:
            w_se = int(round(w_in * se_ratio))
            self.se = SE(w_b, w_se)
        # 1x1
        self.conv3 = nn.Conv2d(w_b, w_out, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(w_out)

        self.shortcut = nn.Sequential()
        if stride != 1 or w_in != w_out:
            self.shortcut = nn.Sequential(
                nn.Conv2d(w_in, w_out,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(w_out)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        if self.with_se:
            out = self.se(out)
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class RegNet(nn.Module):
    def __init__(self, cfg, num_classes=10):
        super(RegNet, self).__init__()
        self.cfg = cfg
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(0)
        self.layer2 = self._make_layer(1)
        self.layer3 = self._make_layer(2)
        self.layer4 = self._make_layer(3)
        self.linear = nn.Linear(self.cfg['widths'][-1], num_classes)

    def _make_layer(self, idx):
        depth = self.cfg['depths'][idx]
        width = self.cfg['widths'][idx]
        stride = self.cfg['strides'][idx]
        group_width = self.cfg['group_width']
        bottleneck_ratio = self.cfg['bottleneck_ratio']
        se_ratio = self.cfg['se_ratio']

        layers = []
        for i in range(depth):
            s = stride if i == 0 else 1
            layers.append(Block(self.in_planes, width,
                                s, group_width, bottleneck_ratio, se_ratio))
            self.in_planes = width
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def RegNetX_200MF():
    cfg = {
        'depths': [1, 1, 4, 7],
        'widths': [24, 56, 152, 368],
        'strides': [1, 1, 2, 2],
        'group_width': 8,
        'bottleneck_ratio': 1,
        'se_ratio': 0,
    }
    return RegNet(cfg)


def RegNetX_400MF():
    cfg = {
        'depths': [1, 2, 7, 12],
        'widths': [32, 64, 160, 384],
        'strides': [1, 1, 2, 2],
        'group_width': 16,
        'bottleneck_ratio': 1,
        'se_ratio': 0,
    }
    return RegNet(cfg)


def RegNetY_400MF():
    cfg = {
        'depths': [1, 2, 7, 12],
        'widths': [32, 64, 160, 384],
        'strides': [1, 1, 2, 2],
        'group_width': 16,
        'bottleneck_ratio': 1,
        'se_ratio': 0.25,
    }
    return RegNet(cfg)


def test():
    net = RegNetX_200MF()
    print(net)
    x = torch.randn(2, 3, 32, 32)
    y = net(x)
    print(y.shape)


if __name__ == '__main__':
    test()


### `modules/pytorch_cifar/models/resnet.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/resnet.py
'''ResNet in PyTorch.

For Pre-activation ResNet, see 'preact_resnet.py'.

Reference:
[1] Kaiming He, Xiangyu Zhang, Shaoqing Ren, Jian Sun
    Deep Residual Learning for Image Recognition. arXiv:1512.03385
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, self.expansion *
                               planes, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion*planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512*block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def ResNet18():
    return ResNet(BasicBlock, [2, 2, 2, 2])


def ResNet34():
    return ResNet(BasicBlock, [3, 4, 6, 3])


def ResNet50():
    return ResNet(Bottleneck, [3, 4, 6, 3])


def ResNet101():
    return ResNet(Bottleneck, [3, 4, 23, 3])


def ResNet152():
    return ResNet(Bottleneck, [3, 8, 36, 3])


def test():
    net = ResNet18()
    y = net(torch.randn(1, 3, 32, 32))
    print(y.size())

# test()


### `modules/pytorch_cifar/models/resnext.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/resnext.py
'''ResNeXt in PyTorch.

See the paper "Aggregated Residual Transformations for Deep Neural Networks" for more details.
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    '''Grouped convolution block.'''
    expansion = 2

    def __init__(self, in_planes, cardinality=32, bottleneck_width=4, stride=1):
        super(Block, self).__init__()
        group_width = cardinality * bottleneck_width
        self.conv1 = nn.Conv2d(in_planes, group_width, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(group_width)
        self.conv2 = nn.Conv2d(group_width, group_width, kernel_size=3, stride=stride, padding=1, groups=cardinality, bias=False)
        self.bn2 = nn.BatchNorm2d(group_width)
        self.conv3 = nn.Conv2d(group_width, self.expansion*group_width, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion*group_width)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*group_width:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*group_width, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*group_width)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNeXt(nn.Module):
    def __init__(self, num_blocks, cardinality, bottleneck_width, num_classes=10):
        super(ResNeXt, self).__init__()
        self.cardinality = cardinality
        self.bottleneck_width = bottleneck_width
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(num_blocks[0], 1)
        self.layer2 = self._make_layer(num_blocks[1], 2)
        self.layer3 = self._make_layer(num_blocks[2], 2)
        # self.layer4 = self._make_layer(num_blocks[3], 2)
        self.linear = nn.Linear(cardinality*bottleneck_width*8, num_classes)

    def _make_layer(self, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(Block(self.in_planes, self.cardinality, self.bottleneck_width, stride))
            self.in_planes = Block.expansion * self.cardinality * self.bottleneck_width
        # Increase bottleneck_width by 2 after each stage.
        self.bottleneck_width *= 2
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        # out = self.layer4(out)
        out = F.avg_pool2d(out, 8)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def ResNeXt29_2x64d():
    return ResNeXt(num_blocks=[3,3,3], cardinality=2, bottleneck_width=64)

def ResNeXt29_4x64d():
    return ResNeXt(num_blocks=[3,3,3], cardinality=4, bottleneck_width=64)

def ResNeXt29_8x64d():
    return ResNeXt(num_blocks=[3,3,3], cardinality=8, bottleneck_width=64)

def ResNeXt29_32x4d():
    return ResNeXt(num_blocks=[3,3,3], cardinality=32, bottleneck_width=4)

def test_resnext():
    net = ResNeXt29_2x64d()
    x = torch.randn(1,3,32,32)
    y = net(x)
    print(y.size())

# test_resnext()


### `modules/pytorch_cifar/models/senet.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/senet.py
'''SENet in PyTorch.

SENet is the winner of ImageNet-2017. The paper is not released yet.
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

        # SE layers
        self.fc1 = nn.Conv2d(planes, planes//16, kernel_size=1)  # Use nn.Conv2d instead of nn.Linear
        self.fc2 = nn.Conv2d(planes//16, planes, kernel_size=1)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        # Squeeze
        w = F.avg_pool2d(out, out.size(2))
        w = F.relu(self.fc1(w))
        w = F.sigmoid(self.fc2(w))
        # Excitation
        out = out * w  # New broadcasting feature from v0.2!

        out += self.shortcut(x)
        out = F.relu(out)
        return out


class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super(PreActBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)

        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False)
            )

        # SE layers
        self.fc1 = nn.Conv2d(planes, planes//16, kernel_size=1)
        self.fc2 = nn.Conv2d(planes//16, planes, kernel_size=1)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))

        # Squeeze
        w = F.avg_pool2d(out, out.size(2))
        w = F.relu(self.fc1(w))
        w = F.sigmoid(self.fc2(w))
        # Excitation
        out = out * w

        out += shortcut
        return out


class SENet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(SENet, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block,  64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def SENet18():
    return SENet(PreActBlock, [2,2,2,2])


def test():
    net = SENet18()
    y = net(torch.randn(1,3,32,32))
    print(y.size())

# test()


### `modules/pytorch_cifar/models/shufflenet.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/shufflenet.py
'''ShuffleNet in PyTorch.

See the paper "ShuffleNet: An Extremely Efficient Convolutional Neural Network for Mobile Devices" for more details.
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class ShuffleBlock(nn.Module):
    def __init__(self, groups):
        super(ShuffleBlock, self).__init__()
        self.groups = groups

    def forward(self, x):
        '''Channel shuffle: [N,C,H,W] -> [N,g,C/g,H,W] -> [N,C/g,g,H,w] -> [N,C,H,W]'''
        N,C,H,W = x.size()
        g = self.groups
        return x.view(N,g,C//g,H,W).permute(0,2,1,3,4).reshape(N,C,H,W)


class Bottleneck(nn.Module):
    def __init__(self, in_planes, out_planes, stride, groups):
        super(Bottleneck, self).__init__()
        self.stride = stride

        mid_planes = out_planes/4
        g = 1 if in_planes==24 else groups
        self.conv1 = nn.Conv2d(in_planes, mid_planes, kernel_size=1, groups=g, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_planes)
        self.shuffle1 = ShuffleBlock(groups=g)
        self.conv2 = nn.Conv2d(mid_planes, mid_planes, kernel_size=3, stride=stride, padding=1, groups=mid_planes, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_planes)
        self.conv3 = nn.Conv2d(mid_planes, out_planes, kernel_size=1, groups=groups, bias=False)
        self.bn3 = nn.BatchNorm2d(out_planes)

        self.shortcut = nn.Sequential()
        if stride == 2:
            self.shortcut = nn.Sequential(nn.AvgPool2d(3, stride=2, padding=1))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.shuffle1(out)
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        res = self.shortcut(x)
        out = F.relu(torch.cat([out,res], 1)) if self.stride==2 else F.relu(out+res)
        return out


class ShuffleNet(nn.Module):
    def __init__(self, cfg):
        super(ShuffleNet, self).__init__()
        out_planes = cfg['out_planes']
        num_blocks = cfg['num_blocks']
        groups = cfg['groups']

        self.conv1 = nn.Conv2d(3, 24, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(24)
        self.in_planes = 24
        self.layer1 = self._make_layer(out_planes[0], num_blocks[0], groups)
        self.layer2 = self._make_layer(out_planes[1], num_blocks[1], groups)
        self.layer3 = self._make_layer(out_planes[2], num_blocks[2], groups)
        self.linear = nn.Linear(out_planes[2], 10)

    def _make_layer(self, out_planes, num_blocks, groups):
        layers = []
        for i in range(num_blocks):
            stride = 2 if i == 0 else 1
            cat_planes = self.in_planes if i == 0 else 0
            layers.append(Bottleneck(self.in_planes, out_planes-cat_planes, stride=stride, groups=groups))
            self.in_planes = out_planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def ShuffleNetG2():
    cfg = {
        'out_planes': [200,400,800],
        'num_blocks': [4,8,4],
        'groups': 2
    }
    return ShuffleNet(cfg)

def ShuffleNetG3():
    cfg = {
        'out_planes': [240,480,960],
        'num_blocks': [4,8,4],
        'groups': 3
    }
    return ShuffleNet(cfg)


def test():
    net = ShuffleNetG2()
    x = torch.randn(1,3,32,32)
    y = net(x)
    print(y)

# test()


### `modules/pytorch_cifar/models/shufflenetv2.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/shufflenetv2.py
'''ShuffleNetV2 in PyTorch.

See the paper "ShuffleNet V2: Practical Guidelines for Efficient CNN Architecture Design" for more details.
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class ShuffleBlock(nn.Module):
    def __init__(self, groups=2):
        super(ShuffleBlock, self).__init__()
        self.groups = groups

    def forward(self, x):
        '''Channel shuffle: [N,C,H,W] -> [N,g,C/g,H,W] -> [N,C/g,g,H,w] -> [N,C,H,W]'''
        N, C, H, W = x.size()
        g = self.groups
        return x.view(N, g, C//g, H, W).permute(0, 2, 1, 3, 4).reshape(N, C, H, W)


class SplitBlock(nn.Module):
    def __init__(self, ratio):
        super(SplitBlock, self).__init__()
        self.ratio = ratio

    def forward(self, x):
        c = int(x.size(1) * self.ratio)
        return x[:, :c, :, :], x[:, c:, :, :]


class BasicBlock(nn.Module):
    def __init__(self, in_channels, split_ratio=0.5):
        super(BasicBlock, self).__init__()
        self.split = SplitBlock(split_ratio)
        in_channels = int(in_channels * split_ratio)
        self.conv1 = nn.Conv2d(in_channels, in_channels,
                               kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv2 = nn.Conv2d(in_channels, in_channels,
                               kernel_size=3, stride=1, padding=1, groups=in_channels, bias=False)
        self.bn2 = nn.BatchNorm2d(in_channels)
        self.conv3 = nn.Conv2d(in_channels, in_channels,
                               kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(in_channels)
        self.shuffle = ShuffleBlock()

    def forward(self, x):
        x1, x2 = self.split(x)
        out = F.relu(self.bn1(self.conv1(x2)))
        out = self.bn2(self.conv2(out))
        out = F.relu(self.bn3(self.conv3(out)))
        out = torch.cat([x1, out], 1)
        out = self.shuffle(out)
        return out


class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DownBlock, self).__init__()
        mid_channels = out_channels // 2
        # left
        self.conv1 = nn.Conv2d(in_channels, in_channels,
                               kernel_size=3, stride=2, padding=1, groups=in_channels, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv2 = nn.Conv2d(in_channels, mid_channels,
                               kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_channels)
        # right
        self.conv3 = nn.Conv2d(in_channels, mid_channels,
                               kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(mid_channels)
        self.conv4 = nn.Conv2d(mid_channels, mid_channels,
                               kernel_size=3, stride=2, padding=1, groups=mid_channels, bias=False)
        self.bn4 = nn.BatchNorm2d(mid_channels)
        self.conv5 = nn.Conv2d(mid_channels, mid_channels,
                               kernel_size=1, bias=False)
        self.bn5 = nn.BatchNorm2d(mid_channels)

        self.shuffle = ShuffleBlock()

    def forward(self, x):
        # left
        out1 = self.bn1(self.conv1(x))
        out1 = F.relu(self.bn2(self.conv2(out1)))
        # right
        out2 = F.relu(self.bn3(self.conv3(x)))
        out2 = self.bn4(self.conv4(out2))
        out2 = F.relu(self.bn5(self.conv5(out2)))
        # concat
        out = torch.cat([out1, out2], 1)
        out = self.shuffle(out)
        return out


class ShuffleNetV2(nn.Module):
    def __init__(self, net_size):
        super(ShuffleNetV2, self).__init__()
        out_channels = configs[net_size]['out_channels']
        num_blocks = configs[net_size]['num_blocks']

        self.conv1 = nn.Conv2d(3, 24, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(24)
        self.in_channels = 24
        self.layer1 = self._make_layer(out_channels[0], num_blocks[0])
        self.layer2 = self._make_layer(out_channels[1], num_blocks[1])
        self.layer3 = self._make_layer(out_channels[2], num_blocks[2])
        self.conv2 = nn.Conv2d(out_channels[2], out_channels[3],
                               kernel_size=1, stride=1, padding=0, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels[3])
        self.linear = nn.Linear(out_channels[3], 10)

    def _make_layer(self, out_channels, num_blocks):
        layers = [DownBlock(self.in_channels, out_channels)]
        for i in range(num_blocks):
            layers.append(BasicBlock(out_channels))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        # out = F.max_pool2d(out, 3, stride=2, padding=1)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.relu(self.bn2(self.conv2(out)))
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


configs = {
    0.5: {
        'out_channels': (48, 96, 192, 1024),
        'num_blocks': (3, 7, 3)
    },

    1: {
        'out_channels': (116, 232, 464, 1024),
        'num_blocks': (3, 7, 3)
    },
    1.5: {
        'out_channels': (176, 352, 704, 1024),
        'num_blocks': (3, 7, 3)
    },
    2: {
        'out_channels': (224, 488, 976, 2048),
        'num_blocks': (3, 7, 3)
    }
}


def test():
    net = ShuffleNetV2(net_size=0.5)
    x = torch.randn(3, 3, 32, 32)
    y = net(x)
    print(y.shape)


# test()


### `modules/pytorch_cifar/models/vgg.py`


In [ ]:
%%writefile modules/pytorch_cifar/models/vgg.py
'''VGG11/13/16/19 in Pytorch.'''
import torch
import torch.nn as nn


cfg = {
    'VGG11': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG13': [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG16': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'],
    'VGG19': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M', 512, 512, 512, 512, 'M', 512, 512, 512, 512, 'M'],
}


class VGG(nn.Module):
    def __init__(self, vgg_name):
        super(VGG, self).__init__()
        self.features = self._make_layers(cfg[vgg_name])
        self.classifier = nn.Linear(512, 10)

    def forward(self, x):
        out = self.features(x)
        out = out.view(out.size(0), -1)
        out = self.classifier(out)
        return out

    def _make_layers(self, cfg):
        layers = []
        in_channels = 3
        for x in cfg:
            if x == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                layers += [nn.Conv2d(in_channels, x, kernel_size=3, padding=1),
                           nn.BatchNorm2d(x),
                           nn.ReLU(inplace=True)]
                in_channels = x
        layers += [nn.AvgPool2d(kernel_size=1, stride=1)]
        return nn.Sequential(*layers)


def test():
    net = VGG('VGG11')
    x = torch.randn(2,3,32,32)
    y = net(x)
    print(y.size())

# test()


### `schemas/generate_labels.toml`


In [ ]:
%%writefile schemas/generate_labels.toml
###
# generate_labels schema
# From input expert training trajectories, produces FLIPped labels.
# Outputs the poisoned labels, true labels, and losses as .npy files.
###

[generate_labels]
input_pths = "string. Format string path to model checkpoint .pth files with three '{}'s."
opt_pths = "string. Format string path to optimizer checkpoint .pth files with three '{}'s."
output_dir = "string. Path to output directory (slurm compatible)."
expert_model = "string: {r32p, r18, r18_tin, vgg, vgg_pretrain, vit_pretrain}. For ResNets, VGG-19s, and ViTs."
dataset = "string: {cifar, cifar_100, tiny_imagenet}. For CIFAR-10, CIFAR-100 and Tiny Imagenet datasets."
trainer = "string: {sgd, adam}. Specifies optimizer."
source_label = "int: {-1,0,...,9}. Specifies label to mimic. -1 indicates all labels."
target_label = "int: {0,1,...,9}. Specifies label to attack."
poisoner = "string: Form: {{1,2,3,9}xp, {1,2}xs, {1,4}xl}. Integer resembles number of attacks and string represents type."

[OPTIONAL]
batch_size = "int: {0,1,...,infty}. Specifies batch size. Set to default for trainer if omitted."
epochs = "int: {0,1,...,infty}. Specifies number of epochs. Set to default for trainer if omitted."
train_pct = "float: [0, 1]. Specifies percentage of dataset available to attacker. Set to 1 by default."
lambda = "float: [0, infty]. Specifies regularization parameter. Set to 0 by default."
expert_config = "dict. Specifies expert checkpoints. Set to default if omitted. See example_attack."
attack_config = "dict. Specifies algorithm parameters. Set to default if ommited. See example_attack."

### `schemas/select_flips.toml`


In [ ]:
%%writefile schemas/select_flips.toml
### 
# select_flips schema
# Given a set of poisoned labels, computes margins and produces FLIPs.
# Outputs coalesced labels for each budget and the true labels.
###

[select_flips]
budgets = "list. Integer list of flip budgets to compute labels for."
input_label_glob = "string. glob path to model checkpoint .pth files with three '{}'s (slurm compatible)."
true_labels = "string. Path to true label .npy file (slurm compatible)."
output_dir = "string. Path to output directory (slurm compatible)."


### `schemas/train_expert.toml`


In [ ]:
%%writefile schemas/train_expert.toml
### 
# train_expert schema
# Records trajectories for an expert model.
# Outputs the .pth files for expert and optimizer trajectories. 
###

[train_expert]
output_dir = "string. Path to output directory (slurm compatible)."
model = "string: {r32p, r18, r18_tin, vgg, vgg_pretrain, vit_pretrain}. For ResNets, VGG-19s, and ViTs."
dataset = "string: {cifar, cifar_100, tiny_imagenet}. For CIFAR-10, CIFAR-100 and Tiny Imagenet datasets."
trainer = "string: {sgd, adam}. Specifies optimizer."
source_label = "int: {-1,0,...,9}. Specifies label to mimic. -1 indicates all labels."
target_label = "int: {0,1,...,9}. Specifies label to attack."
poisoner = "string: Form: {{1,2,3,9}xp, {1,2}xs, {1,4}xl}. Integer resembles number of attacks and string represents type."
checkpoint_iters = "int: {0,1,...,infty}. Number of iterations between each checkpoint record."

[OPTIONAL]
batch_size = "int: {0,1,...,infty}. Specifies batch size. Set to default for trainer if omitted."
epochs = "int: {0,1,...,infty}. Specifies number of epochs. Set to default for trainer if omitted."
train_pct = "float: [0, 1]. Specifies percentage of dataset available to attacker. Set to 1 by default."
optim_kwargs = "dict. Optional keywords for Pytorch SGD / Adam optimizer. See sever example."
scheduler_kwargs = "dict. Optional keywords for Pytorch learning rate optimizer (with SGD). See sever example."

### `schemas/train_user.toml`


In [ ]:
%%writefile schemas/train_user.toml
### 
# train_user schema
# Trains and records metrics on a downstream model trained on input labels.
# Outputs the poison accuracy, clean accuracy, and training labels .npy files and a final model .pth.
###

[train_user]
input_labels = "string. Path to input labels .npy files (slurm compatible)."
output_dir = "string. Path to output directory (slurm compatible)."
user_model = "string: {r32p, r18, r18_tin, vgg, vgg_pretrain, vit_pretrain}. For ResNets, VGG-19s, and ViTs."
dataset = "string: {cifar, cifar_100, tiny_imagenet}. For CIFAR-10, CIFAR-100 and Tiny Imagenet datasets."
trainer = "string: {sgd, adam}. Specifies optimizer."
source_label = "int: {0,1,...,9}. Specifies label to mimic."
target_label = "int: {0,1,...,9}. Specifies label to attack."
poisoner = "string: Form: {{1,2,3,9}xp, {1,2}xs, {1,4}xl}. Integer resembles number of attacks and string represents type."

[OPTIONAL]
true_labels = "string. Path to input labels .npy files (slurm compatible)."
soft = "bool. Specifies whether to compute on logit or hard labels."
alpha = "float: [0, 1]. Specifies interpolation parameter between true (1) and input (0) labels. Set to 0 (full input) if omitted."
batch_size = "int: {0,1,...,infty}. Specifies batch size. Set to default for trainer if omitted."
epochs = "int: {0,1,...,infty}. Specifies number of epochs. Set to default for trainer if omitted."
optim_kwargs = "dict. Optional keywords for Pytorch SGD / Adam optimizer. See sever example."
scheduler_kwargs = "dict. Optional keywords for Pytorch learning rate optimizer (with SGD). See sever example."

### `experiments/example_attack/config.toml`


In [ ]:
%%writefile experiments/example_attack/config.toml
# This example trains a single expert and generates poisoned labels
# for the sinusoidal (1xs) trigger with ResNet-32s. The labels are 
# FLIPped at the provided budgets. The config file is broken down 
# into three modules detailed in the schemas/ folder.

# Module to train and record an expert trajectory.
[train_expert]
output_dir = "out/checkpoints/r32p_1xs/0/"
model = "r32p"
trainer = "sgd"
dataset = "cifar"
source_label = 9
target_label = 4
poisoner = "1xs"
epochs = 20
checkpoint_iters = 50

# Module to generate attack labels from the expert trajectories.
[generate_labels]
input_pths = "out/checkpoints/r32p_1xs/{}/model_{}_{}.pth"
opt_pths = "out/checkpoints/r32p_1xs/{}/model_{}_{}_opt.pth"
expert_model = "r32p"
trainer = "sgd"
dataset = "cifar"
source_label = 9
target_label = 4
poisoner = "1xs"
output_dir = "experiments/example_attack/"
lambda = 0.0

[generate_labels.expert_config]
experts = 1
min = 0
max = 20

[generate_labels.attack_config]
iterations = 5
one_hot_temp = 5
alpha = 0
label_kwargs = {lr = 150, momentum = 0.5}

# Module to flip labels at the provided budgets.
[select_flips]
budgets = [150, 300, 500, 1000, 1500]
input_label_glob = "experiments/example_attack/labels.npy"
true_labels = "experiments/example_attack/true.npy"
output_dir = "experiments/example_attack/"

### `experiments/example_downstream/config.toml`


In [ ]:
%%writefile experiments/example_downstream/config.toml
# This example trains a user model on the poisoned labels from
# example_attack with 1500 budget and records the attack metrics.
# The config file is broken down into a single module detailed in
# the schemas/ folder.

# Module to train a user model on input labels.
[train_user]
input_labels = "experiments/example_attack/1500.npy"
user_model = "r32p"
trainer = "sgd"
dataset = "cifar"
source_label = 9
target_label = 4
poisoner = "1xs"
output_dir = "experiments/example_downstream/"
soft = false
alpha = 0.0

### `experiments/example_downstream_soft/config.toml`


In [ ]:
%%writefile experiments/example_downstream_soft/config.toml
# This example trains a user model on the (soft) logits from
# example_attack and records the attack metrics. The config file 
# is broken down into a single module detailed in the schemas/ folder.

# Module to train a user model on input labels.
[train_user]
input_labels = "experiments/example_attack/labels.npy"
true_labels = "experiments/example_attack/true.npy"
user_model = "r32p"
trainer = "sgd"
dataset = "cifar"
source_label = 9
target_label = 4
poisoner = "1xs"
output_dir = "experiments/example_downstream/"
soft = true
alpha = 0.2


### `experiments/example_precomputed/config.toml`


In [ ]:
%%writefile experiments/example_precomputed/config.toml
# This example trains a user model on precomputed labels with a 1500
# flip budget. The config file is broken down into a single module
# detailed in the schemas/ folder.

# Module to train a user model on input labels.
[train_user]
input_labels = "precomputed_labels/cifar/r32p/1xs/1500.npy"
user_model = "r32p"
trainer = "sgd"
dataset = "cifar"
source_label = 9
target_label = 4
poisoner = "1xs"
output_dir = "experiments/example_precomputed/"
soft = false
alpha = 0.0

### `experiments/example_precomputed_mix/config.toml`


In [ ]:
%%writefile experiments/example_precomputed_mix/config.toml
# This example trains a ViT user model on precomputed ResNet labels
# with a 1500 flip budget. The config file is broken down into a 
# single module detailed in the schemas/ folder.

# Module to train a user model on input labels.
[train_user]
input_labels = "precomputed_labels/cifar/r32p/1xs/1500.npy"
user_model = "vit-pretrain"
trainer = "sgd"
dataset = "cifar"
source_label = 9
target_label = 4
poisoner = "1xs"
output_dir = "experiments/example_precomputed_mix/"
soft = false
alpha = 0.0

[train_user.optim_kwargs]
lr = 0.01
weight_decay = 0.0002

### `README.md`


In [ ]:
%%writefile README.md
# FLIP
## tl;dr
Official implementation of [FLIP](https://arxiv.org/abs/2310.18933), presented at [NeurIPS 2023](https://neurips.cc/virtual/2023/poster/70392). The implementation is a cleaned-up 'fork' of the [backdoor-suite](https://github.com/SewoongLab/backdoor-suite). Precomputed labels for our main table are available [here](https://github.com/SewoongLab/FLIP/releases/). More details are available in the paper. A more complete (messy) version of the code is available upon request.

**Authors:** [Rishi D. Jha\*](http://rishijha.com/), Jonathan Hayase\*, Sewoong Oh

---
## Abstract
In a backdoor attack, an adversary injects corrupted data into a model's training dataset in order to gain control over its predictions on images with a specific attacker-defined trigger. A typical corrupted training example requires altering both the image, by applying the trigger, and the label. Models trained on clean images, therefore, were considered safe from backdoor attacks. However, in some common machine learning scenarios, the training labels are provided by potentially malicious third-parties. This includes crowd-sourced annotation and knowledge distillation. We, hence, investigate a fundamental question: can we launch a successful backdoor attack by only corrupting labels? We introduce a novel approach to design label-only backdoor attacks, which we call FLIP, and demonstrate its strengths on three datasets (CIFAR-10, CIFAR-100, and Tiny-ImageNet) and four architectures (ResNet-32, ResNet-18, VGG-19, and Vision Transformer). With only 2\% of CIFAR-10 labels corrupted, FLIP achieves a near-perfect attack success rate of $99.4\%$ while suffering only a $1.8\%$ drop in the clean test accuracy. Our approach builds upon the recent advances in trajectory matching, originally introduced for dataset distillation.

![Diagram of algorithm.](/img/flip.png)

---

## In this repo

This repo is split into three main folders: `experiments`, `modules`, and `schemas`. The `experiments` folder (as described in more detail [here](#installation)) contains subfolders and `.toml` configuration files on which an experiment may be run. The `modules` folder stores source code for each of the subsequent part of an experiment. These modules take in specific inputs and outputs as defined by their subseqeunt `.toml` documentation in the `schemas` folder. Each module refers to a step of the FLIP algorithm.

Additionally, in the [Precomputed Labels](https://github.com/SewoongLab/FLIP/releases/) release, labels used for the main table of our paper are provided for analysis.

Please don't hesitate to file a GitHub issue or reach out for any issues or requests!

### Existing modules:
1. `base_utils`: Utility module, used by the base modules.
1. `train_expert`: Step 1 of our algorithm: training expert models and recording trajectories.
1. `generate_labels`: Step 2 of our algorithm: generating poisoned labels from trajectories.
1. `select_flips`: Step 3 of algorithm: strategically flipping labels within some budget.
1. `train_user`: Evaluation module to assess attack success rate.

More documentation can be found in the `schemas` folder.

### Supported Datasets:
1. CIFAR-10
1. CIFAR-100
1. Tiny ImageNet

---
## Installation
### Prerequisites:
The prerequisite packages are stored in `requirements.txt` and can be installed using pip:
```
pip install -r requirements.txt
```
Or conda:
```
conda install --file requirements.txt
```
Note that the requirements encapsulate our testing enviornments and may be unnecessarily tight! Any relevant updates to the requirements are welcomed.

## Running An Experiment
### Setting up:
To initialize an experiment, create a subfolder in the `experiments` folder with the name of your experiment:
```
mkdir experiments/[experiment name]
```
In that folder initialize a config file called `config.toml`. An example can be seen here: `experiments/example_attack/config.toml`.

The `.toml` file should contain references to the modules that you would like to run with each relevant field as defined by its documentation in `schemas/[module name]`. This file will serve as the configuration file for the entire experiment. As a convention the output for module **n** is the input for module **n + 1**.

**Note:** the `[INTERNAL]` block of a schema should not be transferred into a config file.

```
[module_name_1]
output=...
field2=...
...
fieldn=...

[module_name_2]
input=...
output=...
...
fieldn=...

...

[module_name_k]
input=...
field2=...
...
fieldn=...
```

### Running a module:
At the moment, all experiments must be manually run using:
```
python run_experiment.py [experiment name]
```
The experiment will automatically pick up on the configuration provided by the file. 

As an example, to run the `example_attack` experiment one could run:
```
python run_experiment.py example_attack
```
More module documentation can be found in the `schemas` folder.



### `LICENSE`


In [ ]:
%%writefile LICENSE
MIT License

Copyright (c) 2021 Rishi Dev Jha, Jonathan Hayase, and Sewoong Oh.

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

### `.gitignore`


In [ ]:
%%writefile .gitignore
# Project specific
out/
out*/
slurm/
reps/
*.pth
*.npz
.vscode
/migrated
/nbconfig
results/
viz/
utils/
precomputed_labels/

# Byte-compiled / optimized / DLL files
__pycache__/
*.py[cod]
*$py.class

# C extensions
*.so

# Distribution / packaging
.Python
build/
develop-eggs/
dist/
downloads/
eggs/
.eggs/
lib/
lib64/
parts/
sdist/
var/
wheels/
share/python-wheels/
*.egg-info/
.installed.cfg
*.egg
MANIFEST

# PyInstaller
#  Usually these files are written by a python script from a template
#  before PyInstaller builds the exe, so as to inject date/other infos into it.
*.manifest
*.spec

# Installer logs
pip-log.txt
pip-delete-this-directory.txt

# Unit test / coverage reports
htmlcov/
.tox/
.nox/
.coverage
.coverage.*
.cache
nosetests.xml
coverage.xml
*.cover
*.py,cover
.hypothesis/
.pytest_cache/
cover/

# Translations
*.mo
*.pot

# Django stuff:
*.log
local_settings.py
db.sqlite3
db.sqlite3-journal

# Flask stuff:
instance/
.webassets-cache

# Scrapy stuff:
.scrapy

# Sphinx documentation
docs/_build/

# PyBuilder
.pybuilder/
target/

# Jupyter Notebook
.ipynb_checkpoints

# IPython
profile_default/
ipython_config.py

# pyenv
#   For a library or package, you might want to ignore these files since the code is
#   intended to run in multiple environments; otherwise, check them in:
# .python-version

# pipenv
#   According to pypa/pipenv#598, it is recommended to include Pipfile.lock in version control.
#   However, in case of collaboration, if having platform-specific dependencies or dependencies
#   having no cross-platform support, pipenv may install dependencies that don't work, or not
#   install all needed dependencies.
#Pipfile.lock

# PEP 582; used by e.g. github.com/David-OConnor/pyflow
__pypackages__/

# Celery stuff
celerybeat-schedule
celerybeat.pid

# SageMath parsed files
*.sage.py

# Environments
.env
.venv
env/
venv/
ENV/
env.bak/
venv.bak/

# Spyder project settings
.spyderproject
.spyproject

# Rope project settings
.ropeproject

# mkdocs documentation
/site

# mypy
.mypy_cache/
.dmypy.json
dmypy.json

# Pyre type checker
.pyre/

# pytype static type analyzer
.pytype/

# Cython debug symbols
cython_debug/

### `.gitmodules`


In [ ]:
%%writefile .gitmodules
[submodule "modules/pytorch_cifar"]
	path = modules/pytorch_cifar
	url = https://github.com/kuangliu/pytorch-cifar
	ignore = all

## 4. Run an experiment

By default this runs the `example_attack` experiment exactly as the original
`python run_experiment.py example_attack` invocation does locally.

Change the experiment name below to run any of the example experiments:
`example_attack`, `example_downstream`, `example_downstream_soft`,
`example_precomputed`, or `example_precomputed_mix`.


In [ ]:
import sys, os

# Ensure we are at the project root before running the experiment.
os.chdir(PROJECT_DIR)

EXPERIMENT_NAME = 'example_attack'

# Invoke the original entry point exactly as on local: `python run_experiment.py <name>`
!python run_experiment.py {EXPERIMENT_NAME}
